# 04 · Qwen + LoRA: replay of the 27B runs, a look inside the adapters, and a small stand-in re-run

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheridanLu/detecting-the-detector/blob/main/notebooks/04_lora_qwen.ipynb)

> **What is replayed, what is re-run, what is a stand-in**
>
> | part | what happens | status |
> |---|---|---|
> | 1. Qwen3.8-27B + LoRA results | false-flag rates, F1, seed spread, recomputed from the saved per-essay scores in `data/preds/qwen/` | **REPLAYED from the A40 run, not re-run** |
> | 2. Look inside the 8 adapters | the trained LoRA files (Hugging Face) are opened and counted; the 27B base model is never loaded | **inspection only** (needs read access to a private Hugging Face repo) |
> | 3. Mini-LoRA demo | the same LoRA recipe, same texts, arms A and C, trained here on the small public model Qwen3-0.6B | **RE-RUN, but on a STAND-IN model**: its numbers are not the 27B result |
> | 4. REAL Qwen3.8-27B (off by default) | downloads the public FP8 base model, converts it to bf16 as on our server, attaches the archived adapters, scores essays and compares with the archived scores; optional re-training | **NOT executed by us**: needs a GPU with >= 80 GB (Colab Pro: RTX PRO 6000 96 GB, A100/H100 80 GB); only its no-GPU pre-flight checks run by default |
>
> The 27B model cannot run on a free T4 or a laptop: its base model (the public `Qwen/Qwen3.8-27B-FP8`, converted once
> from FP8 to bf16) needs about 52 GB of GPU memory, and each training run took about 4 hours on two 46 GB GPUs.
> Part 4 runs it on a big Colab Pro GPU; **we have not executed Part 4 on such a GPU.**

| Mode | Part 1 (replay) | Part 2 (adapters) | Part 3 (stand-in Qwen3-0.6B) | Hardware |
|---|---|---|---|---|
| SMOKE | full | headers of all 8 files; no download | arms A, C x 1 seed, 80 training essays, 64 tokens, 5 training steps, 20 test essays per corpus | CPU, a few minutes |
| FAST (default) | full | headers of all 8 files; no download | arms A, C x 1 seed, 2,000 training essays each, 256 tokens, Liang test set + LOCNESS | T4 GPU. Estimate (not measured): roughly 10-30 min |
| FULL | full | headers + download of `seed0_A_n4000` (499 MB) | arms A, C x 2 seeds, all 8,000 training essays, 512 tokens (the 27B setting), all 14,272 essays scored | T4 GPU. Estimate (not measured): several hours, roughly 3-10 h |

The GPU times are estimates from model size and number of tokens; this notebook has **not** been run on a GPU yet.
During training it prints a projected training time after 1, 5 and 20 steps, so you see the real figure early.
The outputs saved in this file come from a **SMOKE run on a laptop CPU** (with the adapter download switched on for
testing); they are not GPU results.

In [1]:
# --- Setup: find the repository (or download it on Colab) and install anything Colab does not have ---
import os, sys, time, pathlib, subprocess
T0 = time.time()
REPO_URL = "https://github.com/SheridanLu/detecting-the-detector"

def _is_repo(p):
    return (p / "ddlib" / "__init__.py").is_file() and (p / "data").is_dir()

def _locate_or_fetch():
    here = pathlib.Path.cwd().resolve()
    for p in [pathlib.Path(os.environ.get("DD_REPO", here)), here, *here.parents]:
        if _is_repo(p):
            return p                                   # mode 1: already inside the repo
    dest = pathlib.Path("/content" if os.path.isdir("/content") else here) / "detecting-the-detector"
    if not _is_repo(dest) and not dest.exists():       # mode 2: git clone (token optional, for a private repo)
        tok = os.environ.get("GH_TOKEN")
        try:
            from google.colab import userdata
            tok = userdata.get("GH_TOKEN") or tok
        except Exception:
            pass
        url = REPO_URL.replace("https://", f"https://x-access-token:{tok}@") if tok else REPO_URL
        subprocess.run(["git", "clone", "--depth", "1", "-q", url, str(dest)], env=dict(os.environ, GIT_TERMINAL_PROMPT="0"))
        if tok and dest.exists():
            subprocess.run(["git", "-C", str(dest), "remote", "set-url", "origin", REPO_URL])   # token not kept on disk
    if _is_repo(dest):
        return dest
    return None                                        # mode 3 (zip upload) is handled by ddlib below

root = _locate_or_fetch()
if root is None:
    # Zip fallback: needs the small ddlib/paths.py helper, which we cannot import yet, so do it here.
    print("Could not find or clone the repository.\n"
          "Fix 1: add a Colab secret GH_TOKEN (key icon, left sidebar) with read access, allow notebook access, re-run.\n"
          f"Fix 2: download the repo as a zip ({REPO_URL} > Code > Download ZIP) and upload it when asked.")
    from google.colab import files
    import zipfile
    up = files.upload()
    zipfile.ZipFile(next(k for k in up if k.endswith(".zip"))).extractall("/content")
    root = next(p.parent.parent for p in pathlib.Path("/content").glob("*/ddlib/__init__.py") if _is_repo(p.parent.parent))
sys.path.insert(0, str(root))
os.chdir(root)                                         # all paths below are relative to the repo root

_req = [l.strip() for l in open("requirements-colab.txt") if l.strip() and not l.startswith("#")]
if _req:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_req], check=True)

from ddlib import get_repo
REPO = get_repo()
print("Python", sys.version.split()[0])

# Notebook 04 also needs PyTorch, transformers, peft, accelerate (Colab has torch and transformers; peft is added).
import importlib.util, importlib.metadata
_need = [m for m in ("torch", "transformers", "peft", "accelerate", "safetensors", "huggingface_hub")
         if importlib.util.find_spec(m) is None]
if not _need:
    _tv = tuple(int(x) for x in importlib.metadata.version("transformers").split(".")[:2])
    if _tv < (4, 56):
        _need.append("newer transformers")             # Qwen3 sequence classification + the `dtype=` argument
    elif not os.path.isdir(os.path.join(importlib.util.find_spec("transformers").submodule_search_locations[0], "models", "qwen3_5")):
        _need.append("transformers with Qwen3.5 classes (Part 4)")   # checked on disk, without importing transformers
if _need:
    print("Installing:", ", ".join(_need), "(from requirements-nb04.txt)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-nb04.txt"], check=True)
for _m in ("torch", "transformers", "peft", "huggingface_hub"):
    print(f"{_m} {importlib.metadata.version(_m)}", end="   ")
print()

Repository found. Data folder present: True
Python 3.12.16
torch 2.15.0   transformers 5.23.0   peft 0.21.3   huggingface_hub 2.4.0   


## Configuration

In [2]:
# --- CONFIG ---------------------------------------------------------------------------------------
# MODE = "SMOKE" (minutes on a CPU; tiny version used for automated testing), "FAST" (default, needs a GPU) or "FULL".
# Change the string below to switch. (The DD_MODE environment variable is only used by automated tests.)
MODE = os.environ.get("DD_MODE", "FAST").upper()
assert MODE in ("SMOKE", "FAST", "FULL"), MODE

# Part 2: look inside the archived adapters (needs read access to the private Hugging Face repo, see Part 2)
CHECK_ADAPTER_HEADERS = True                     # read only the header of each of the 8 files (~100 kB each)
DOWNLOAD_ADAPTERS = MODE == "FULL"               # download one adapter file (seed0_A_n4000, 499 MB) and open it
DOWNLOAD_ALL_ADAPTERS = False                    # True: download all 8 (3.99 GB)
_env = os.environ.get("DD_DOWNLOAD_ADAPTERS")    # automated tests only: 0 / 1 / all
if _env is not None:
    DOWNLOAD_ADAPTERS = _env.lower() in ("1", "true", "all"); DOWNLOAD_ALL_ADAPTERS = _env.lower() == "all"

# Part 3: stand-in LoRA run (same recipe as the 27B runs, small public model)
STANDIN_MODEL = os.environ.get("DD_STANDIN_MODEL", "Qwen/Qwen3-0.6B")
STANDIN_ARMS = ["A", "C"]
STANDIN = {"SMOKE": dict(seeds=[0], n_per_class=40, max_len=64, score_set="core", per_corpus=20),
           "FAST":  dict(seeds=[0], n_per_class=1000, max_len=256, score_set="core", per_corpus=None),
           "FULL":  dict(seeds=[0, 1], n_per_class=4000, max_len=512, score_set="all", per_corpus=None)}[MODE]
STANDIN_FP32 = False      # True: plain fp32 on the GPU (slower). Use it only if the fp16 run on a T4 reports NaN losses.

# Where finished stand-in runs are saved. A finished run is skipped when the notebook is run again.
OUT_DIR = pathlib.Path(os.environ.get("DD_NB04_OUT", "outputs/nb04_standin"))
SAVE_TO_DRIVE = False     # Colab: True keeps finished runs in your Google Drive, so they survive a disconnect
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = pathlib.Path("/content/drive/MyDrive/detecting-the-detector/nb04_standin")
OUT_DIR.mkdir(parents=True, exist_ok=True)

from ddlib.paths import rel
print(f"MODE = {MODE} | adapter headers: {CHECK_ADAPTER_HEADERS} | download adapters: "
      f"{'all 8' if DOWNLOAD_ADAPTERS and DOWNLOAD_ALL_ADAPTERS else 'seed0_A_n4000' if DOWNLOAD_ADAPTERS else 'no'}")
print(f"stand-in: {STANDIN_MODEL}, arms {STANDIN_ARMS}, seeds {STANDIN['seeds']}, {2 * STANDIN['n_per_class']:,} "
      f"training essays per run, max {STANDIN['max_len']} tokens | results folder: {rel(OUT_DIR)}")
TIMES = {}

MODE = SMOKE | adapter headers: True | download adapters: seed0_A_n4000
stand-in: Qwen/Qwen3-0.6B, arms ['A', 'C'], seeds [0], 80 training essays per run, max 64 tokens | results folder: standin_default


## What is LoRA? (for non-experts)

A large language model such as Qwen3.8-27B has about 26 billion numbers ("weights") that were learned in
pre-training. Re-training all of them for our task (is this essay written by AI?) would need far more GPU memory than
we had. **LoRA (low-rank adaptation)** keeps all those weights **frozen** and adds a small **add-on** next to most of
the model's internal layers. Each add-on is a pair of thin matrices, A and B, whose product B x A is added to the frozen
layer's output. "Rank 16" means the add-on is only 16 numbers "wide", so it is tiny compared with the layer it
modifies. On top, a small **classifier head** turns the model's final internal state into two scores: human or AI.
Only the add-ons and the head are trained.

Settings used for the 27B runs (`a40/train_qwen.py`), copied unchanged into the stand-in in Part 3:

| setting | value |
|---|---|
| task | sequence classification, 2 classes (PEFT `task_type="SEQ_CLS"`) |
| LoRA rank r / scaling alpha / dropout | 16 / 32 / 0.05 |
| which layers get an add-on | every linear layer (`target_modules="all-linear"`); the classifier head is trained in full |
| trainable numbers (27B) | 124,741,120 = 0.476% of the model |
| training | 1 pass over 8,000 essays (4,000 human + 4,000 AI), learning rate 1e-4, batch 8 x 2 accumulation steps, seed = run seed |
| text | the same `clean()` as every other detector, cut at 512 tokens |
| threshold | an essay is *flagged as AI* if the AI probability is at least 0.5 |

These settings were chosen once and **not tuned** (no search over learning rate, rank, or length).

## Part 1 · REPLAYED from the A40 run, not re-run

> **REPLAYED from the A40 run, not re-run.** Nothing is trained or run in this part. The numbers are recomputed from
> the per-essay scores the 27B detector gave in the original run (`data/preds/qwen/`, 8 files = 4 arms x 2 seeds,
> 14,272 essays each) with the same `ddlib.metrics` code as notebook 05, and checked against the tables the original
> run archived in `data/results/`.

**Why only 2 seeds for the 27B?** A *seed* fixes the random choices of a training run (order of the essays, starting
values of the add-ons). Running the same setting with different seeds shows how much a result depends on luck. One
27B run took about 4 hours on two GPUs, so 4 arms x 2 seeds already took about 32 GPU-pair hours; logistic regression
and RoBERTa got 5 seeds. With 2 seeds we can see *that* the results move, but not reliably *how much*.

In [3]:
t_part = time.time()
import json
import numpy as np, pandas as pd
from ddlib import data as D, metrics as M, lora_qwen as LQ
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

train_ids = D.train_exam_ids()
qpreds = {(arm, s): D.load_preds("Qwen27B", arm, s) for arm in "ABCD" for s in D.SEEDS["Qwen27B"]}
rows, prf_rows = [], []
for (arm, s), p in qpreds.items():
    rows.append(dict(model="Qwen27B", arm=arm, seed=s, **M.run_row(p, train_ids)))
    prf_rows.append(dict(model="Qwen27B", arm=arm, run=D.run_name(arm, s), **M.liang_prf_from_scores(p)))
QR, QP = pd.DataFrame(rows), pd.DataFrame(prf_rows)
QR["f1"] = QP.f1.values

per_seed = pd.DataFrame({
    "TOEFL flagged %": QR.FP_TOEFL * 100, "familiar native (Hewlett) %": QR.FP_Hewlett * 100,
    "unfamiliar native %": QR.FP_nativeOOD_mean * 100, "F1 (Liang set)": QR.f1,
    "bias AUC TOEFL vs Hewlett": QR.bias_TOEFLvsHewlett, "TOEFL flagged at 90%-det. thr. %": QR.flag90_TOEFL * 100})
per_seed.index = pd.MultiIndex.from_frame(QR[["arm", "seed"]])
print("REPLAYED from the A40 run, not re-run. Qwen3.8-27B + LoRA, per seed, threshold 0.5 unless stated.")
display(per_seed.round(3))
print(f"{len(qpreds)} saved score files, {len(next(iter(qpreds.values()))):,} essays each.")

REPLAYED from the A40 run, not re-run. Qwen3.8-27B + LoRA, per seed, threshold 0.5 unless stated.


TOEFL flagged %  familiar native (Hewlett) %  unfamiliar native %  F1 (Liang set)  bias AUC TOEFL vs Hewlett  TOEFL flagged at 90%-det. thr. %
arm seed                                                                                                                                                
A   0              56.044                        5.682               20.233           0.854                      0.948                             1.099
    1              23.077                        6.818               38.322           0.805                      0.892                             1.099
B   0              30.769                        4.545               17.869           0.887                      0.942                             1.099
    1              28.571                        3.409               39.517           0.794                      0.949                             1.099
C   0               4.396                        4.545                2.315           0.976                      0.637                             1.099
    1               5.495                        3.409               15.905           0.910                      0.636                             1.099
D   0              38.462                        9.091                4.236           0.922                      0.907                             3.297
    1              30.769                        2.273                9.419           0.913                      0.971                             1.099

8 saved score files, 14,272 essays each.


### Seed spread
Same arm, same data, only the seed differs. A large gap means a single 27B run should not be trusted on its own.
The last column is the 90%-detection threshold: the 27B scores are almost all 0 or 1, so this threshold lands near 1.0
and flags almost nothing; that is why, as decided before the follow-up runs (`a40/PREREG_followup_gpu.md`), the 27B is
read at threshold 0.5.

In [4]:
spread = []
for arm in "ABCD":
    g = QR[QR.arm == arm].set_index("seed")
    for col, name, k in (("FP_TOEFL", "TOEFL flagged %", 100), ("FP_nativeOOD_mean", "unfamiliar native %", 100), ("f1", "F1", 1)):
        spread.append(dict(arm=arm, measure=name, seed0=g.loc[0, col] * k, seed1=g.loc[1, col] * k,
                           mean=g[col].mean() * k, gap_between_seeds=abs(g.loc[0, col] - g.loc[1, col]) * k,
                           SD=g[col].std() * k))
spread = pd.DataFrame(spread).set_index(["arm", "measure"])
print("REPLAYED from the A40 run, not re-run.")
display(spread.round(3))
a = spread.loc[("A", "TOEFL flagged %")]
print(f"Example: arm A flags {a['mean']:.0f}% of TOEFL essays on average, but the two seeds give {a['seed0']:.0f}% and "
      f"{a['seed1']:.0f}% (gap {a['gap_between_seeds']:.0f} points).")

REPLAYED from the A40 run, not re-run.


seed0   seed1    mean  gap_between_seeds      SD
arm measure                                                               
A   TOEFL flagged %      56.044  23.077  39.560             32.967  23.311
    unfamiliar native %  20.233  38.322  29.278             18.089  12.791
    F1                    0.854   0.805   0.830              0.049   0.035
B   TOEFL flagged %      30.769  28.571  29.670              2.198   1.554
    unfamiliar native %  17.869  39.517  28.693             21.649  15.308
    F1                    0.887   0.794   0.840              0.093   0.066
C   TOEFL flagged %       4.396   5.495   4.945              1.099   0.777
    unfamiliar native %   2.315  15.905   9.110             13.589   9.609
    F1                    0.976   0.910   0.943              0.067   0.047
D   TOEFL flagged %      38.462  30.769  34.615              7.692   5.439
    unfamiliar native %   4.236   9.419   6.828              5.182   3.664
    F1                    0.922   0.913   0.917              0.009   0.006

Example: arm A flags 40% of TOEFL essays on average, but the two seeds give 56% and 23% (gap 33 points).


### Checks against the archived tables
Every replayed number is compared with the table the original run wrote (`data/results/`). These are the same saved
scores and the same formulas, so the differences should be at the level of floating-point rounding (tolerance 1e-12).
The last block checks the rounded numbers quoted in the presentation.

In [5]:
TOL = 1e-12
checks = []        # (check, max_abs_diff, result)
def add(name, frame):
    d = float(frame.max_abs_diff.max()); ok = bool((frame.result == "PASS").all()) and len(frame) > 0
    checks.append(dict(check=name, rows=int(frame.rows_compared.min()), columns=len(frame), max_abs_diff=d, result="PASS" if ok else "FAIL"))

# 1. arm means (+ SD of TOEFL over seeds) vs model_comparison_all.csv
cols = list(M.FP_COLS) + list(M.TPR_COLS) + ["AUC_ArguGPTvsTOEFL", "AUC_Liang352vsHewlett", "bias_TOEFLvsHewlett", "bias_TOEFLvsNativeOOD", "FP_nativeOOD_mean"]
CM = QR.groupby(["model", "arm"], sort=False)[cols].mean()
CM["TOEFL_sd"] = QR.groupby(["model", "arm"], sort=False).FP_TOEFL.std()
CM["n_seeds"] = QR.groupby(["model", "arm"], sort=False).size()
old = D.load_result("model_comparison_all"); old = old[old.model == "Qwen27B"]
add("arm means vs model_comparison_all.csv", M.compare_frames(CM.reset_index(), old, cols + ["TOEFL_sd", "n_seeds"], ["model", "arm"], tol=TOL))

# 2. every run vs followup_runs_qwen.csv
o = D.load_result("followup_runs_qwen").rename(columns={
    "P1_toefl_flag90": "flag90_TOEFL", "bias_auc_vs_hewlett": "bias_TOEFLvsHewlett", "P3_arguGPT_vs_toefl": "AUC_ArguGPTvsTOEFL",
    "native_ood_mean90": "flag90_nativeOOD_mean", "bias_auc_vs_native_ood": "bias_TOEFLvsNativeOOD", "thr": "thr_tpr90",
    "flag05_TOEFL_real_91": "FP_TOEFL", "flag05_HewlettStudentEssay_real_88": "FP_Hewlett", "P3_ai352_vs_hewlett": "AUC_Liang352vsHewlett"})
add("per run vs followup_runs_qwen.csv", M.compare_frames(QR, o, ["thr_tpr90", "FP_TOEFL", "FP_Hewlett", "flag90_TOEFL", "flag90_nativeOOD_mean",
                                                                 "bias_TOEFLvsHewlett", "bias_TOEFLvsNativeOOD", "AUC_ArguGPTvsTOEFL", "AUC_Liang352vsHewlett"], ["arm", "seed"], tol=TOL))

# 3. precision / recall / F1 per run vs prf1_runs.csv, and arm means / SDs vs prf1_summary.csv
keys = ["accuracy", "precision", "recall", "f1", "false_flag_rate", "f1_bal_TOEFL", "prec_bal_TOEFL", "f1_bal_Hewlett", "prec_bal_Hewlett"]
add("per run vs prf1_runs.csv", M.compare_frames(QP, D.load_result("prf1_runs"), keys + ["n_human", "n_ai"], ["model", "run"], tol=TOL))
ps = QP.groupby(["model", "arm"])[keys].agg(["mean", "std"]); ps.columns = [f"{a}__{b}" for a, b in ps.columns]
add("arm means and SDs vs prf1_summary.csv", M.compare_frames(ps.reset_index(), D.load_result("prf1_summary"), list(ps.columns), ["model", "arm"], tol=TOL))

# 4. original metrics script (arms A/B only) vs metrics_runs.csv
mr = D.load_result("metrics_runs"); mr = mr[mr.model == "Qwen27B"]
ab = []
for (arm, s), p in qpreds.items():
    if arm in "AB":
        by = M.by_corpus(p)
        ab.append(dict(run=D.run_name(arm, s), thr_tpr90=M.thr_tpr90(p), bias_TOEFL_vs_Hewlett=M.auc(by[M.TOEFL], by[M.HEWLETT]),
                       rate_t05_TOEFL_real_91=M.flag_rate(by[M.TOEFL]), rate_t05_HewlettStudentEssay_real_88=M.flag_rate(by[M.HEWLETT]),
                       det_AI352_vs_LiangHuman2=M.auc(np.concatenate([by[c].to_numpy() for c in M.AI352]),
                                                      np.r_[by[M.TOEFL].to_numpy(), by[M.HEWLETT].to_numpy()])))
add("arms A/B vs metrics_runs.csv", M.compare_frames(pd.DataFrame(ab), mr, ["thr_tpr90", "bias_TOEFL_vs_Hewlett", "rate_t05_TOEFL_real_91",
                                                                           "rate_t05_HewlettStudentEssay_real_88", "det_AI352_vs_LiangHuman2"], ["run"], tol=TOL))

# 5. paired test on the 91 TOEFL essays vs followup_paired_qwen.csv (same random-number order as notebook 05)
rng = np.random.default_rng(0); flags = {}
for (arm, s), p in qpreds.items():
    pp = M.drop_train_exams(p, train_ids)
    flags[(arm, s)] = (pp[pp.corpus == M.TOEFL].set_index("essay_id").score >= M.thr_tpr90(pp)).astype(int)
PAIR = pd.DataFrame([dict(pair=f"{x}-{y}", seed=s, **M.paired_flags(flags[(x, s)], flags[(y, s)], n_boot=1000, rng=rng))
                     for x, y in (("C", "A"), ("C", "D"), ("B", "A")) for s in D.SEEDS["Qwen27B"]])
add("paired tests vs followup_paired_qwen.csv", M.compare_frames(PAIR, D.load_result("followup_paired_qwen"),
                                                                  ["rate_x", "rate_y", "diff", "lo", "hi", "p"], ["pair", "seed"], tol=TOL))
CHECKS_REPLAY = pd.DataFrame(checks)
display(CHECKS_REPLAY)

# 6. the rounded numbers quoted in the presentation (pass = rounds to the quoted value)
g = lambda arm, s=None: QR[(QR.arm == arm) & ((QR.seed == s) if s is not None else True)]
quoted = [("TOEFL flagged, arm A (mean)", g("A").FP_TOEFL.mean(), 0.40, 0.005), ("TOEFL flagged, arm A seed 0", g("A", 0).FP_TOEFL.mean(), 0.56, 0.005),
          ("TOEFL flagged, arm A seed 1", g("A", 1).FP_TOEFL.mean(), 0.23, 0.005), ("TOEFL flagged, arm B", g("B").FP_TOEFL.mean(), 0.30, 0.005),
          ("TOEFL flagged, arm C", g("C").FP_TOEFL.mean(), 0.05, 0.005), ("TOEFL flagged, arm D", g("D").FP_TOEFL.mean(), 0.35, 0.005),
          ("F1 Liang set, arm A", g("A").f1.mean(), 0.83, 0.005), ("F1 Liang set, arm C", g("C").f1.mean(), 0.94, 0.005)]
QUOTED = pd.DataFrame([M.compare(n, v, q, t) for n, v, q, t in quoted])
display(QUOTED.round(4))
n_pass = (CHECKS_REPLAY.result == "PASS").sum() + (QUOTED.result == "PASS").sum()
n_all = len(CHECKS_REPLAY) + len(QUOTED)
print(f"REPLAY CHECKS: {n_pass} PASS, {n_all - n_pass} FAIL (largest difference to the archived tables: {CHECKS_REPLAY.max_abs_diff.max():.1e})")
TIMES["Part 1 replay"] = time.time() - t_part

,check,rows,columns,max_abs_diff,result
0,arm means vs model_comparison_all.csv,4,22,2.775558e-17,PASS
1,per run vs followup_runs_qwen.csv,8,9,0.000000e+00,PASS
2,per run vs prf1_runs.csv,8,11,0.000000e+00,PASS
3,arm means and SDs vs prf1_summary.csv,4,18,0.000000e+00,PASS
4,arms A/B vs metrics_runs.csv,4,5,0.000000e+00,PASS
5,paired tests vs followup_paired_qwen.csv,6,6,0.000000e+00,PASS


,check,recomputed,archived,tolerance,result
0,"TOEFL flagged, arm A (mean)",0.3956,0.40,0.005,PASS
1,"TOEFL flagged, arm A seed 0",0.5604,0.56,0.005,PASS
2,"TOEFL flagged, arm A seed 1",0.2308,0.23,0.005,PASS
3,"TOEFL flagged, arm B",0.2967,0.30,0.005,PASS
4,"TOEFL flagged, arm C",0.0495,0.05,0.005,PASS
5,"TOEFL flagged, arm D",0.3462,0.35,0.005,PASS
6,"F1 Liang set, arm A",0.8299,0.83,0.005,PASS
7,"F1 Liang set, arm C",0.9430,0.94,0.005,PASS


REPLAY CHECKS: 14 PASS, 0 FAIL (largest difference to the archived tables: 2.8e-17)


**Reading the replay.** Training on exam-style non-native essays (arm C) brings the 27B's TOEFL false flags down to
about 5%, and its F1 on the Liang test set up; the control arm D (more native essays of another kind) helps much less.
But the two seeds of the same arm can differ a lot (arm A: see the seed-spread table), and the pre-registered paired
test at the 90%-detection threshold shows no difference at all, because that threshold flags almost nothing for this
model. Treat the 27B results as indicative, as stated in the pre-registration.

## Part 2 · Look inside the trained adapters (optional)

The 8 trained LoRA add-ons (one per arm and seed) are stored on Hugging Face in the **private** repository
`SheridanLu/detecting-the-detector-qwen-lora` (folders `seed0_A_n4000`, ..., `seed1_D`; each has
`adapter_model.safetensors`, 499 MB, and `adapter_config.json`). To run this part you need read access to that repo
and a Hugging Face token: on Colab add a secret named `HF_TOKEN` (key icon in the left sidebar, then switch on
*Notebook access*); locally set the environment variable `HF_TOKEN` or log in once with `hf auth login`. The token is
never printed or saved by this notebook. Without a token the part is skipped and the rest of the notebook still runs.

**This is a "look inside", not a re-run.** A `.safetensors` file starts with a small table of contents (the *header*:
name, shape and number type of every stored matrix). Reading it tells us exactly what was trained: how many numbers,
in which layers, with which rank. We never load the 27B base model; without it the adapters cannot score any essay.
**Using the adapters for inference needs the 27B base model** (the public `Qwen/Qwen3.8-27B-FP8`, converted from FP8
to bf16 as on our server) and a GPU with >= 80 GB. That is Part 4 (off by default); Parts 1-3 replay the saved scores
instead.

In [6]:
t_part = time.time()
TOKEN, TOKEN_SOURCE = LQ.hf_token()
print("Hugging Face token:", f"found ({TOKEN_SOURCE}); it is not printed" if TOKEN else "none found")
HEADERS = {}
if not CHECK_ADAPTER_HEADERS:
    print("SKIPPED: CHECK_ADAPTER_HEADERS = False.")
elif TOKEN is None:
    print("SKIPPED: no Hugging Face token (see the text above). Parts 1 and 3 do not need one.")
else:
    for run in LQ.ADAPTER_RUNS:
        try:
            HEADERS[run] = LQ.tensor_table(LQ.remote_safetensors_header(f"{run}/adapter_model.safetensors", TOKEN))
        except Exception as e:
            print(f"{run}: header could not be read ({type(e).__name__}: {str(e).splitlines()[0][:120]})")
    if HEADERS:
        hdr = pd.DataFrame([dict(adapter=r, tensors=len(t), lora_pairs=int((t.kind == "lora_A").sum()),
                                 lora_rank=",".join(map(str, LQ.lora_rank_from_shapes(t))), parameters=int(t.params.sum()),
                                 equals_124_741_120="PASS" if int(t.params.sum()) == LQ.EXPECTED_TRAINABLE else "FAIL")
                            for r, t in HEADERS.items()])
        print("Read from the file headers on the Hub (HTTP range requests, no tensor data downloaded):")
        display(hdr)
TIMES["Part 2a headers"] = time.time() - t_part

Hugging Face token: found (existing Hugging Face login on this machine); it is not printed


Read from the file headers on the Hub (HTTP range requests, no tensor data downloaded):


,adapter,tensors,lora_pairs,lora_rank,parameters,equals_124_741_120
0,seed0_A_n4000,1213,606,16,124741120,PASS
1,seed0_B_n4000,1213,606,16,124741120,PASS
2,seed0_C,1213,606,16,124741120,PASS
3,seed0_D,1213,606,16,124741120,PASS
4,seed1_A_n4000,1213,606,16,124741120,PASS
5,seed1_B_n4000,1213,606,16,124741120,PASS
6,seed1_C,1213,606,16,124741120,PASS
7,seed1_D,1213,606,16,124741120,PASS


In [7]:
t_part = time.time()
LOCAL = {}
if not DOWNLOAD_ADAPTERS:
    print("SKIPPED: DOWNLOAD_ADAPTERS = False (default in SMOKE and FAST). Set it to True in the CONFIG cell to download "
          "seed0_A_n4000 (499 MB); DOWNLOAD_ALL_ADAPTERS = True downloads all 8 (3.99 GB).")
elif TOKEN is None:
    print("SKIPPED: no Hugging Face token.")
else:
    from huggingface_hub import hf_hub_download
    for run in (LQ.ADAPTER_RUNS if DOWNLOAD_ALL_ADAPTERS else ["seed0_A_n4000"]):
        t = time.time()
        st = hf_hub_download(LQ.ADAPTER_REPO, f"{run}/adapter_model.safetensors", token=TOKEN)
        cf = hf_hub_download(LQ.ADAPTER_REPO, f"{run}/adapter_config.json", token=TOKEN)
        h = LQ.read_safetensors_header(st); tt = LQ.tensor_table(h)
        LOCAL[run] = dict(path=st, config=json.load(open(cf)), table=tt, header=h)
        expected_size = 8 + h["header_bytes"] + int(tt.nbytes.sum())
        print(f"{run}: downloaded/found in {time.time() - t:.0f} s | file {h['file_bytes']:,} bytes = 8 + header "
              f"{h['header_bytes']:,} + tensor data {int(tt.nbytes.sum()):,} -> {'PASS' if expected_size == h['file_bytes'] else 'FAIL'} | "
              f"parameters {int(tt.params.sum()):,} -> {'PASS' if int(tt.params.sum()) == LQ.EXPECTED_TRAINABLE else 'FAIL'} (expected 124,741,120)")
TIMES["Part 2b download"] = time.time() - t_part

seed0_A_n4000: downloaded/found in 0 s | file 499,125,968 bytes = 8 + header 181,960 + tensor data 498,944,000 -> PASS | parameters 124,741,120 -> PASS (expected 124,741,120)


### What the adapter contains
Below: the numbers of the first downloaded adapter (or, without a download, of the header read from the Hub), grouped
by where they sit in the model, then by layer type, then a few individual tensors.

In [8]:
src = next(iter(LOCAL.values()))["table"] if LOCAL else (HEADERS.get("seed0_A_n4000") if HEADERS else None)
if src is None:
    print("SKIPPED: no adapter header available (no token, or CHECK_ADAPTER_HEADERS = False).")
else:
    tt = src
    print(f"seed0_A_n4000: {len(tt):,} tensors, {int(tt.params.sum()):,} parameters in total "
          f"({'PASS' if int(tt.params.sum()) == LQ.EXPECTED_TRAINABLE else 'FAIL'}: expected 124,741,120)")
    print("number types:", tt.groupby("dtype").params.sum().to_dict())
    print("LoRA rank read from the shapes (A is r x in, B is out x r):", LQ.lora_rank_from_shapes(tt))
    display(tt.groupby(["part", "kind"]).agg(tensors=("name", "size"), parameters=("params", "sum")))
    lm = tt[tt.part == "language model"]
    by_mod = lm[lm.kind == "lora_A"].groupby("module").agg(layers=("layer", "nunique"), first=("layer", "min"), last=("layer", "max"))
    by_mod["layer numbers"] = lm[lm.kind == "lora_A"].groupby("module").layer.apply(
        lambda v: ", ".join(map(str, sorted(v)[:6])) + (", ..." if len(v) > 6 else ""))
    by_mod["parameters (A+B)"] = lm.groupby("module").params.sum()
    print(f"Language model: LoRA add-ons in {lm.layer.nunique()} layers (numbers {lm.layer.min()}-{lm.layer.max()}); "
          "which linear layers carry them:")
    display(by_mod)
    vis = tt[tt.part == "vision tower"]
    if len(vis):
        print(f"Vision tower (the base model can also read images): {vis[vis.layer >= 0].layer.nunique()} blocks + merger also got LoRA "
              f"add-ons, {int(vis.params.sum()):,} parameters, because 'all-linear' targets every linear layer.")
    print("A few tensors:")
    with pd.option_context("display.max_colwidth", 100):
        display(tt[["name", "kind", "shape", "dtype", "params"]].iloc[[0, 1, 2, 3, len(tt) - 1]])

seed0_A_n4000: 1,213 tensors, 124,741,120 parameters in total (PASS: expected 124,741,120)
number types: {'BF16': 10240, 'F32': 124730880}
LoRA rank read from the shapes (A is r x in, B is out x r): [16]


tensors  parameters
part            kind                                       
classifier head head (trained in full)        1       10240
language model  lora_A                      496    54263808
                lora_B                      496    62464000
vision tower    lora_A                      110     3499776
                lora_B                      110     4503296

Language model: LoRA add-ons in 64 layers (numbers 0-63); which linear layers carry them:


,layers,first,last,layer numbers,parameters (A+B)
module,,,,,
linear_attn.in_proj_a,48,0,62,"0, 1, 2, 4, 5, 6, ...",3969024
linear_attn.in_proj_b,48,0,62,"0, 1, 2, 4, 5, 6, ...",3969024
linear_attn.in_proj_qkv,48,0,62,"0, 1, 2, 4, 5, 6, ...",11796480
linear_attn.in_proj_z,48,0,62,"0, 1, 2, 4, 5, 6, ...",8650752
linear_attn.out_proj,48,0,62,"0, 1, 2, 4, 5, 6, ...",8650752
mlp.down_proj,64,0,63,"0, 1, 2, 3, 4, 5, ...",23068672
mlp.gate_proj,64,0,63,"0, 1, 2, 3, 4, 5, ...",23068672
mlp.up_proj,64,0,63,"0, 1, 2, 3, 4, 5, ...",23068672
self_attn.k_proj,16,3,63,"3, 7, 11, 15, 19, 23, ...",1572864


Vision tower (the base model can also read images): 27 blocks + merger also got LoRA add-ons, 8,003,072 parameters, because 'all-linear' targets every linear layer.
A few tensors:


,name,kind,shape,dtype,params
0,base_model.model.score.weight,head (trained in full),"(2, 5120)",BF16,10240
1,base_model.model.model.language_model.layers.0.linear_attn.in_proj_a.lora_A.weight,lora_A,"(16, 5120)",F32,81920
2,base_model.model.model.language_model.layers.0.linear_attn.in_proj_a.lora_B.weight,lora_B,"(48, 16)",F32,768
3,base_model.model.model.language_model.layers.0.linear_attn.in_proj_b.lora_A.weight,lora_A,"(16, 5120)",F32,81920
1212,base_model.model.model.visual.blocks.26.mlp.linear_fc2.lora_B.weight,lora_B,"(1152, 16)",F32,18432


In [9]:
if not LOCAL:
    print("SKIPPED: no adapter downloaded (DOWNLOAD_ADAPTERS = False or no token).")
else:
    run, L = next(iter(LOCAL.items()))
    t = time.time()
    vals = LQ.tensor_value_checks(L["path"], L["table"])        # reads the adapter's own matrices, not the base model
    lora = vals[vals.kind.str.startswith("lora")]
    print(f"{run}: numerical rank of every LoRA matrix (A: 16 x in, B: out x 16) took {time.time() - t:.0f} s")
    display(lora.groupby(["part", "kind"]).agg(matrices=("name", "size"), all_zero=("all_zero", "sum"),
                                               min_rank=("numerical_rank", "min"), max_rank=("numerical_rank", "max")))
    zero_b = lora[(lora.kind == "lora_B") & lora.all_zero]
    if len(zero_b):
        untouched = int(vals[vals.name.isin(zero_b.name.str.replace("lora_B", "lora_A")) | vals.name.isin(zero_b.name)].params.sum())
        print(f"{len(zero_b)} lora_B matrices are exactly zero, all in: {sorted(zero_b.part.unique())}. LoRA starts every B at "
              f"zero, so these add-ons never changed during training: {untouched:,} of the {int(vals.params.sum()):,} 'trainable' "
              f"parameters ({untouched / vals.params.sum():.1%}) did nothing; {int(vals.params.sum()) - untouched:,} were actually trained.")
    else:
        print("No lora_B matrix is all zero: every add-on was changed by training.")
    cfg = dict(L["config"])
    print("\nadapter_config.json (as saved by PEFT; 'base_model_name_or_path' is the folder on the training server):")
    print(json.dumps({k: cfg[k] for k in ("peft_type", "task_type", "r", "lora_alpha", "lora_dropout", "target_modules",
                                          "modules_to_save", "bias", "use_dora", "use_rslora", "peft_version",
                                          "base_model_name_or_path") if k in cfg}, indent=1))
    print("other keys (not set by our recipe):", ", ".join(k for k in cfg if k not in ("peft_type", "task_type", "r", "lora_alpha",
          "lora_dropout", "target_modules", "modules_to_save", "bias", "use_dora", "use_rslora", "peft_version", "base_model_name_or_path")))

seed0_A_n4000: numerical rank of every LoRA matrix (A: 16 x in, B: out x 16) took 2 s


matrices  all_zero  min_rank  max_rank
part           kind                                          
language model lora_A       496         0      16.0      16.0
               lora_B       496         0      16.0      16.0
vision tower   lora_A       110         0      16.0      16.0
               lora_B       110       110       0.0       0.0

110 lora_B matrices are exactly zero, all in: ['vision tower']. LoRA starts every B at zero, so these add-ons never changed during training: 8,003,072 of the 124,741,120 'trainable' parameters (6.4%) did nothing; 116,738,048 were actually trained.

adapter_config.json (as saved by PEFT; 'base_model_name_or_path' is the folder on the training server):
{
 "peft_type": "LORA",
 "task_type": "SEQ_CLS",
 "r": 16,
 "lora_alpha": 32,
 "lora_dropout": 0.05,
 "target_modules": [
  "gate_proj",
  "in_proj_b",
  "linear_fc1",
  "in_proj_qkv",
  "down_proj",
  "linear_fc2",
  "up_proj",
  "out_proj",
  "q_proj",
  "in_proj_z",
  "in_proj_a",
  "o_proj",
  "qkv",
  "v_proj",
  "k_proj",
  "attn.proj"
 ],
 "modules_to_save": [
  "classifier",
  "score"
 ],
 "bias": "none",
 "use_dora": false,
 "use_rslora": false,
 "peft_version": "0.21.3",
 "base_model_name_or_path": "/data/int4096_unified_v4/qwen27b_bf16_from_fp8"
}
other keys (not set by our recipe): alora_invocation_tokens, alpha_pattern, arrow_

**Reading the inspection.** The header confirms the recipe: rank-16 matrix pairs on every linear layer of the language
model (attention and feed-forward layers) plus a fully trained classifier head (`score`, 2 x 5120), 124,741,120 numbers
in total. Because `all-linear` also targets the base model's image encoder ("vision tower"), the count includes add-ons
there too. If the value check above reports all-zero B matrices in the vision tower, those add-ons never received a
training signal (our essays are text only), so the number of parameters that actually changed is smaller than the
headline 0.476%; this does not affect any score, because text never passes through the vision tower.

## Part 3 · Mini-LoRA demo on a STAND-IN model (this part really trains)

> **STAND-IN: Qwen3-0.6B, NOT the 27B result; numbers are not comparable.** Every number in this part comes from a
> model about 45 times smaller than the 27B detector, trained here with fewer essays and shorter texts in SMOKE/FAST.

What is the same as in the 27B runs: the LoRA recipe (PEFT `SEQ_CLS`, r = 16, alpha = 32, dropout 0.05, all linear
layers, classifier head trained, learning rate 1e-4, batch 8 x 2, one pass, seed), the text cleaning `clean()`, the
training files (`manifests.json` for arm A, `followup/trainsets/seed*_C.parquet` for arm C), the scoring code and the
metric code of Part 1. What differs: the base model (public `Qwen/Qwen3-0.6B` instead of the private 27B), and in
SMOKE/FAST the number of training essays and the text length (see the mode table at the top).

In SMOKE/FAST the smaller training sets keep the design of the study: arms A and C get **the same AI essays and the
same native essays for half of their human side**; only the other half differs (more native essays in A, exam-style
non-native essays in C), exactly as in the full training sets.

Number format: the A40 run used bf16. A T4 GPU has no bf16, so on a T4 the weights stay in fp32 and training uses fp16
mixed precision with loss scaling; on a GPU with bf16 (e.g. A100, L4) bf16 is used as on the A40; on a CPU plain fp32.

In [10]:
t_part = time.time()
from huggingface_hub import HfApi
info = HfApi().model_info(STANDIN_MODEL, files_metadata=True)          # public model, no token needed
size = sum(s.size or 0 for s in info.siblings if s.rfilename.endswith(".safetensors"))
print(f"stand-in model on the Hub: {info.id} (revision {info.sha[:10]}), weights {size / 1e9:.2f} GB")
PREC = LQ.pick_precision(STANDIN_FP32)
print("device:", PREC["device"], f"({PREC['gpu']})" if PREC["gpu"] else "", "| number format:", PREC["label"])
if PREC["device"] == "cpu" and MODE != "SMOKE":
    print("WARNING: no GPU. FAST/FULL on a CPU would take many hours. On Colab: Runtime > Change runtime type > T4 GPU.")

stand-in model on the Hub: Qwen/Qwen3-0.6B (revision c1899de289), weights 1.50 GB
device: cpu  | number format: fp32 on CPU


In [11]:
pool = D.load_pool(cleaned=True); manifests = D.load_manifests()
score_all = D.score_frame()
SCORE_DF = LQ.score_subset(score_all, STANDIN["score_set"], STANDIN["per_corpus"])
TRAIN = {}
for s in STANDIN["seeds"]:
    XA, yA = D.load_trainset(D.run_name("A", s), pool_cleaned=pool, manifests=manifests)
    XC, yC = D.load_trainset(D.run_name("C", s))
    partC = pd.read_parquet(D.data_dir() / "followup" / "trainsets" / f"seed{s}_C.parquet").part.tolist()
    sub_A, sub_C = LQ.paired_subsample_A_C(XA, yA, XC, yC, partC, STANDIN["n_per_class"], seed=s)   # also asserts the shared design
    if STANDIN["n_per_class"] >= 4000:
        TRAIN[("A", s)], TRAIN[("C", s)] = (XA, yA), (XC, yC)          # FULL: the complete training sets
    else:
        TRAIN[("A", s)], TRAIN[("C", s)] = sub_A, sub_C
for (arm, s), (X, y) in TRAIN.items():
    print(f"arm {arm} seed {s}: {len(X):,} training essays ({sum(y):,} AI, {len(y) - sum(y):,} human)")
groups = {"TOEFL": [M.TOEFL], "familiar native (Hewlett)": [M.HEWLETT], "unfamiliar native": M.NATIVE_OOD,
          "GPT-3 AI essays (Liang)": M.AI352, "validation AI": sorted(M.AI_VAL)}
print(f"essays to score per run: {len(SCORE_DF):,} (" + ", ".join(f"{k} {SCORE_DF.corpus.isin(v).sum()}" for k, v in groups.items()) + ")")

arm A seed 0: 80 training essays (40 AI, 40 human)
arm C seed 0: 80 training essays (40 AI, 40 human)
essays to score per run: 220 (TOEFL 20, familiar native (Hewlett) 20, unfamiliar native 60, GPT-3 AI essays (Liang) 80, validation AI 40)


In [12]:
STANDIN_PREDS, STANDIN_INFO = {}, {}
for s in STANDIN["seeds"]:
    for arm in STANDIN_ARMS:
        X, y = TRAIN[(arm, s)]
        cfg = dict(model=STANDIN_MODEL, arm=arm, seed=s, n_train=len(X), max_len=STANDIN["max_len"],
                   score_set=STANDIN["score_set"], per_corpus=STANDIN["per_corpus"], n_scored=len(SCORE_DF), force_fp32=STANDIN_FP32)
        rid = LQ.run_id(STANDIN_MODEL, arm, s, len(X), STANDIN["max_len"])
        if LQ.finished(OUT_DIR, rid, cfg):
            inf = json.load(open(OUT_DIR / f"{rid}.json"))
            print(f"SKIP {rid}: finished earlier (trained in {inf['train_s']:.0f} s on {inf['device']}), using its saved scores")
        else:
            print(f"RUN  {rid}: training on {len(X):,} essays ...", flush=True)
            inf = LQ.train_and_score(X, y, SCORE_DF, cfg, OUT_DIR, rid, log_every=50)
            print(f"     done: model load {inf['load_s']:.0f} s, training {inf['train_s']:.0f} s ({inf['optimizer_steps']} steps), "
                  f"scoring {inf['eval_s']:.0f} s | trainable {inf['trainable_params']:,} of {inf['total_params']:,} "
                  f"({inf['trainable_params'] / inf['total_params']:.2%}) | loss first {inf['loss_first']:.3f}, last {inf['loss_last']:.3f}")
        STANDIN_PREDS[(arm, s)] = pd.read_parquet(OUT_DIR / f"{rid}.parquet")
        STANDIN_INFO[(arm, s)] = inf
bad = [k for k, v in STANDIN_INFO.items() if v["nan_scores"] or v["loss_last"] is None or not np.isfinite(v["loss_last"])]
print("NaN check:", "PASS (no NaN loss or score)" if not bad else f"FAIL for {bad}: set STANDIN_FP32 = True in the CONFIG cell and re-run")
TIMES["Part 3 stand-in training + scoring"] = time.time() - t_part

RUN  Qwen3-0.6B_A_seed0_n80_len64: training on 80 essays ...


W1119 00:48:47.479000 37952 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

    Qwen/Qwen3-0.6B loaded in 2 s (fp32 on CPU); its language-model output layer is dropped and a new 2-class head 'score' is created, as for the 27B. Trainable: 10,094,592 of 606,146,560 (1.67%)


    step 1/5  elapsed     4 s  projected training time    20 s


{'loss': '4.29', 'grad_norm': '103.1', 'learning_rate': '0.0001', 'epoch': '0.2'}


... (12 outputs left out)

{'loss': '2.071', 'grad_norm': '49.98', 'learning_rate': '4e-05', 'epoch': '0.8'}


    step 5/5  elapsed    19 s  projected training time    19 s


{'loss': '2.058', 'grad_norm': '44.17', 'learning_rate': '2e-05', 'epoch': '1'}
{'train_runtime': '18.7', 'train_samples_per_second': '4.278', 'train_steps_per_second': '0.267', 'train_loss': '2.703', 'epoch': '1'}


     done: model load 2 s, training 19 s (5 steps), scoring 11 s | trainable 10,094,592 of 606,146,560 (1.67%) | loss first 4.307, last 2.058
NaN check: PASS (no NaN loss or score)


### Stand-in results (same metric code as Part 1)
Threshold 0.5. *Unfamiliar native* = mean of college essays, CS224N and LOCNESS. F1 on the Liang test set (AI =
positive class). In SMOKE only 20 essays per corpus are scored, so one essay moves a rate by 5 points, and
a model trained for 5 steps on 80 essays is not expected to have learned anything: SMOKE only shows that the code runs.

In [13]:
LABEL = f"STAND-IN: {STANDIN_MODEL.split('/')[-1]}, NOT the 27B result; numbers are not comparable."
srows = []
for (arm, s), p in STANDIN_PREDS.items():
    r = M.run_row(p, train_ids); f = M.liang_prf_from_scores(p)
    srows.append({"arm": arm, "seed": s, "TOEFL flagged %": r["FP_TOEFL"] * 100, "familiar native (Hewlett) %": r["FP_Hewlett"] * 100,
                  "unfamiliar native %": r["FP_nativeOOD_mean"] * 100, "AI caught (Liang GPT-3) %": r["TPR_Liang352"] * 100,
                  "F1 (Liang set)": f["f1"], "bias AUC TOEFL vs Hewlett": r["bias_TOEFLvsHewlett"],
                  "n TOEFL": int((p.corpus == M.TOEFL).sum()), "n Liang set": f["n_human"] + f["n_ai"]})
STANDIN_RES = pd.DataFrame(srows).set_index(["arm", "seed"])
print(LABEL)
print(f"(mode {MODE}: {2 * STANDIN['n_per_class']:,} training essays per run, max {STANDIN['max_len']} tokens)")
display(STANDIN_RES.round(3))

STAND-IN: Qwen3-0.6B, NOT the 27B result; numbers are not comparable.
(mode SMOKE: 80 training essays per run, max 64 tokens)


,,TOEFL flagged %,familiar native (Hewlett) %,unfamiliar native %,AI caught (Liang GPT-3) %,F1 (Liang set),bias AUC TOEFL vs Hewlett,n TOEFL,n Liang set
arm,seed,,,,,,,,
A,0,45.0,45.0,41.667,62.50,0.602,0.398,20,160
C,0,40.0,45.0,40.000,61.25,0.601,0.390,20,160


### Side by side, not a reproduction
Left: the archived 27B numbers (Part 1, mean of 2 seeds, REPLAYED). Right: the stand-in trained in this notebook. They
are shown next to each other only so that you can see both; **the stand-in is not a reproduction of the 27B result**
(different model size, and in SMOKE/FAST fewer essays and shorter texts), so no difference between the two columns
says anything about whether the 27B result is right.

In [14]:
measures = [("FP_TOEFL", "TOEFL flagged %", 100), ("FP_Hewlett", "familiar native (Hewlett) %", 100),
            ("FP_nativeOOD_mean", "unfamiliar native %", 100), ("f1", "F1 (Liang set)", 1)]
side = []
for arm in STANDIN_ARMS:
    for col, name, k in measures:
        sv = STANDIN_RES.xs(arm, level="arm")[name]
        side.append({"arm": arm, "measure": name,
                     "Qwen3.8-27B + LoRA (REPLAYED, 2 seeds)": QR[QR.arm == arm][col].mean() * k,
                     f"STAND-IN {STANDIN_MODEL.split('/')[-1]} ({MODE}, {len(sv)} seed{'s' if len(sv) > 1 else ''})": sv.mean()})
SIDE = pd.DataFrame(side).set_index(["arm", "measure"])
print(LABEL)
display(SIDE.round(3))

STAND-IN: Qwen3-0.6B, NOT the 27B result; numbers are not comparable.


Qwen3.8-27B + LoRA (REPLAYED, 2 seeds)  STAND-IN Qwen3-0.6B (SMOKE, 1 seed)
arm measure                                                                                                 
A   TOEFL flagged %                                              39.560                               45.000
    familiar native (Hewlett) %                                   6.250                               45.000
    unfamiliar native %                                          29.278                               41.667
    F1 (Liang set)                                                0.830                                0.602
C   TOEFL flagged %                                               4.945                               40.000
    familiar native (Hewlett) %                                   3.977                               45.000
    unfamiliar native %                                           9.110                               40.000
    F1 (Liang set)                                                0.943                                0.601

### What the stand-in tells, and what it does not
* **It tells** that the LoRA training recipe of the 27B runs is real, complete code: the same PEFT configuration,
  training loop, scoring and metrics run end to end on public hardware, and you can watch the add-ons being trained.
* **It tells** (in FAST/FULL, on a GPU) how a *small* model of the same family behaves with the same data: whether
  training on exam-style non-native essays (arm C) also lowers TOEFL false flags for it.
* **It does not tell** whether the 27B numbers in Part 1 are correct. A 0.6B model can behave differently from a 27B
  model, for better or worse; agreement would not confirm the 27B result and disagreement would not refute it.
* **It does not tell** anything in SMOKE mode: 5 training steps on 80 essays only prove that the code runs.

## Honest limits
* **27B needs a big GPU**: on a T4 or a laptop the 27B results can only be replayed (Part 1) and its adapters inspected
  (Part 2). Part 4 can reproduce them by inference, or re-train, on a GPU with >= 80 GB (Colab Pro); **this has not been
  executed by us on such a GPU**.
* **Only 2 seeds for the 27B**, and they disagree strongly in some arms (arm A: 56% vs 23% TOEFL false flags); the
  pre-registration therefore labels the 27B effects as indicative.
* **Settings not tuned**: rank, learning rate, text length and number of steps were fixed once for all runs.
* **Threshold**: the 90%-detection threshold is degenerate for the 27B (almost no essay is flagged), so its results are
  read at 0.5.
* **Small TOEFL sample**: 91 essays; one essay = 1.1 percentage points.
* **The stand-in may differ**: Qwen3-0.6B results are a different model's results. The GPU times in the mode table are
  estimates; this notebook has been executed on a laptop CPU in SMOKE mode only, not yet on a Colab T4.

In [15]:
summary = [dict(part="1 replay", check=r.check, result=r.result) for r in CHECKS_REPLAY.itertuples()]
summary += [dict(part="1 replay", check=f"quoted: {r.check}", result=r.result) for r in QUOTED.itertuples()]
if HEADERS:
    summary.append(dict(part="2 adapters", check=f"header parameter count = 124,741,120 ({len(HEADERS)} of 8 adapters read)",
                        result="PASS" if len(HEADERS) == 8 and all(int(t.params.sum()) == LQ.EXPECTED_TRAINABLE for t in HEADERS.values()) else "FAIL"))
for run, L in LOCAL.items():
    h, tt = L["header"], L["table"]
    summary.append(dict(part="2 adapters", check=f"{run}: downloaded file, parameter count and file size",
                        result="PASS" if int(tt.params.sum()) == LQ.EXPECTED_TRAINABLE and 8 + h["header_bytes"] + int(tt.nbytes.sum()) == h["file_bytes"] else "FAIL"))
summary.append(dict(part="3 stand-in", check=f"{len(STANDIN_PREDS)} of {len(STANDIN['seeds']) * len(STANDIN_ARMS)} runs finished, every essay scored, no NaN",
                    result="PASS" if len(STANDIN_PREDS) == len(STANDIN["seeds"]) * len(STANDIN_ARMS) and not bad
                    and all(len(p) == len(SCORE_DF) for p in STANDIN_PREDS.values()) else "FAIL"))
SUMMARY = pd.DataFrame(summary)
with pd.option_context("display.max_rows", 50, "display.max_colwidth", 120):
    display(SUMMARY)
print(f"CHECK SUMMARY: {(SUMMARY.result == 'PASS').sum()} PASS, {(SUMMARY.result == 'FAIL').sum()} FAIL "
      f"(Part 2 skipped: {'yes' if not HEADERS and not LOCAL else 'no'})")
for k, v in TIMES.items():
    print(f"  {k}: {v:.0f} s")
print(f"Notebook finished in {time.time() - T0:.0f} s (mode {MODE}, device {PREC['device']}).")

,part,check,result
0,1 replay,arm means vs model_comparison_all.csv,PASS
1,1 replay,per run vs followup_runs_qwen.csv,PASS
2,1 replay,per run vs prf1_runs.csv,PASS
3,1 replay,arm means and SDs vs prf1_summary.csv,PASS
4,1 replay,arms A/B vs metrics_runs.csv,PASS
5,1 replay,paired tests vs followup_paired_qwen.csv,PASS
6,1 replay,"quoted: TOEFL flagged, arm A (mean)",PASS
7,1 replay,"quoted: TOEFL flagged, arm A seed 0",PASS
8,1 replay,"quoted: TOEFL flagged, arm A seed 1",PASS
9,1 replay,"quoted: TOEFL flagged, arm B",PASS


CHECK SUMMARY: 17 PASS, 0 FAIL (Part 2 skipped: no)
  Part 1 replay: 1 s
  Part 2a headers: 6 s
  Part 2b download: 0 s
  Part 3 stand-in training + scoring: 71 s
Notebook finished in 81 s (mode SMOKE, device cpu).


## Part 4 · REAL 27B RUN: needs a GPU with >= 80 GB; NOT executed in our laptop tests

> **REAL 27B RUN: needs a GPU with >= 80 GB; NOT executed in our laptop tests.** This part is **off by default**
> (`RUN_27B = False` in the next cell). Switched off, only the pre-flight checks of section 4.0 run (they need no GPU)
> and every other cell prints why it was skipped. Everything here was tested only on a laptop CPU, with a tiny random
> model of the same architecture standing in for the 27B (see "What has and has not been tested" below).

What it does, with the base model the 8 adapters were trained on (the public, non-gated Hugging Face repo
`Qwen/Qwen3.8-27B-FP8`, pinned to revision `c3a462cc3d`):

| section | what | default when `RUN_27B = True` |
|---|---|---|
| 4.0 pre-flight (no GPU) | conversion formula on synthetic numbers; which of the repo's 1,606 tensors get converted (read from the file headers, no weights); the real 27B architecture built without memory ("meta" device): trainable parameters = 124,741,120 and tensor names = those in the archived adapter; the tolerances of 4b fail on every pair of *different* archived runs | always runs |
| 4a base model | download FP8 (30.9 GB), convert to bf16 shard by shard with the rule of `a40/dequant_fp8_to_bf16.py` (55.6 GB) | on |
| 4b **reproduction by inference** | load the bf16 base on ONE GPU, attach the archived adapters `seed0_A_n4000` and `seed0_C`, score all 925 essays of the Liang test set, compare essay by essay with `data/preds/qwen/` | on (the strongest evidence) |
| 4c re-training | the recipe of `a40/train_qwen.py` on one GPU: FAST (arms A, C, seed 0, 800 essays per class, 256 tokens) or FULL (8,000 essays, 512 tokens) | off (`TRAIN_27B = False`) |

### How to run it on Colab Pro
1. Open this notebook in Colab (badge at the top). *Runtime > Change runtime type*: choose **RTX PRO 6000** (96 GB),
   or **A100 / H100 (80 GB)**; switch on **High-RAM** if offered. A 40 GB A100, an L4 or a T4 is too small.
2. Key icon in the left sidebar (*Secrets*): add `HF_TOKEN` (read access to the private adapter repo
   `SheridanLu/detecting-the-detector-qwen-lora`; the base model itself needs no token) and `GH_TOKEN` (to clone this
   repository while it is private). Switch on *Notebook access* for both.
3. In the Part 4 CONFIG cell below set `RUN_27B = True` (and, if wanted, `TRAIN_27B = True`). Then *Runtime > Run all*.
   Parts 1-3 run first (the CONFIG cell's `MODE = "SMOKE"` makes Part 3 take only a few minutes).
4. Read the printed PASS / FAIL lines and the `PART 4 CHECK SUMMARY` at the end.

### Downloads, disk, memory, time (all ESTIMATES, not measured on these GPUs)
| item | size or estimate | basis |
|---|---|---|
| download | 30.9 GB base model + 1.0 GB for 2 adapters | file sizes on the Hub |
| disk | ~94 GB if the FP8 files are kept, ~64 GB if each FP8 shard is deleted after conversion (`DELETE_FP8 = "auto"` picks this when the disk is smaller) | 55.6 GB bf16 + 30.9 GB FP8 (or one 6 GB shard) + 2 GB margin |
| GPU memory, scoring | ~49 GiB weights + a few GiB for a batch of 16 x 512 tokens: roughly 52-58 GiB | 26.09 billion numbers x 2 bytes |
| GPU memory, training | weights + LoRA (0.5 GB) + gradients and Adam state (1.5 GB) + activations: roughly 55-70 GiB at batch 4 x 512 | measured by the batch probe before training starts |
| download time | ~5-30 min | depends on Colab's network |
| FP8 -> bf16 conversion | ~5-20 min on the GPU (bound by writing 55.6 GB to disk); 30-90 min on a CPU | |
| loading the base model | ~2-10 min per load | 55.6 GB read from local disk |
| 4b scoring, 925 essays x 2 adapters | ~2-10 min per adapter | 0.2 million tokens x 2 x 26 billion operations |
| 4c FAST, per run (2 runs) | ~10-30 min | ~12% of the tokens of a FULL run |
| 4c FULL, per run | ~1-4 h | original: about 4 h per run on 2 x A40 (only one of the two GPUs computes at a time); an A100 / RTX PRO 6000 / H100 has roughly 2-6x the A40's bf16 speed on paper |

### If something goes wrong
* **Out of GPU memory while training**: the batch probe already falls back from 4 to 2 to 1 essays per step (with
  gradient accumulation 4 / 8 / 16, so 16 essays per optimizer step as in the recipe). If even 1 fails, lower
  `TRAIN_27B_SET[...]["max_len"]` (e.g. 384). While scoring, the batch is halved automatically; or lower `REPRO_BATCH`.
* **Disconnects**: a Colab session can disconnect (idle time-out, 24 h limit). If the runtime is still alive, re-run
  all: downloaded and converted shards, finished reproduction scores and finished training runs are skipped, and a
  training run resumes from its last checkpoint. If the runtime was **reset**, its disk is wiped: the base model must
  be downloaded again (~31 GB); set `SAVE_27B_TO_DRIVE = True` to keep results and training checkpoints in Google
  Drive so that they survive.
* **Not enough disk**: the CONFIG cell prints how much is needed; `DELETE_FP8 = True` lowers it to ~64 GB.

### Why the conversion is done this way (4a)
The adapters were trained on the FP8 model converted to bf16 by `a40/dequant_fp8_to_bf16.py`
(W_bf16 = W_fp8 x scale of its 128 x 128 block). The official bf16 repo `Qwen/Qwen3.8-27B` is **not** the same model
numerically (it differs by the ~2.7% FP8 rounding error), so we download the FP8 repo and repeat our conversion. The
conversion is ported into `ddlib.lora_qwen` with the same formula and the same rules (FP8 tensors converted with their
`weight_scale_inv`, the scale tensors dropped, everything else copied unchanged) and writes the same bf16 folder,
which is then loaded exactly as in `a40/train_qwen.py`. We chose to write the bf16 files to disk rather than convert
straight into GPU memory: loading through `from_pretrained` is the code path of the original run (key names, the new
`score` head, dropped `lm_head` / MTP tensors are handled by transformers as on the server), the conversion runs one
shard at a time (low RAM), can resume after a disconnect, and each shard is checked against the CRC32 list published
in the repo. When the disk is small, each FP8 shard is deleted as soon as it is no longer needed. The multiplication
runs on the GPU when there is one (much faster); 4a checks on the first shard that GPU and CPU give bit-identical
results.

### What has and has not been tested
Tested on a laptop CPU only: the conversion formula (synthetic numbers, including shapes that are not multiples of
128), the tensor selection on the real repo's headers, the real architecture on the meta device, and the whole code
path of 4a-4c on a tiny random model of the same architecture (FP8 checkpoint -> conversion -> loading -> LoRA ->
adapter save and load -> a few training steps -> scoring -> metrics). **Not tested**: loading the real 27B weights,
bf16 on Blackwell (RTX PRO 6000) or Hopper GPUs, the archived adapters on the real 27B, the real memory use, the
out-of-memory fallbacks on a real GPU (only simulated), and every time estimate above.

In [16]:
# --- Part 4 CONFIG: the REAL Qwen3.8-27B. Off by default. -----------------------------------------------------------
RUN_27B = os.environ.get("DD_RUN_27B", "0") == "1"       # True: run 4a + 4b on this machine (needs >= 80 GB GPU)
TRAIN_27B = os.environ.get("DD_TRAIN_27B", "0") == "1"   # True: also re-train (4c)
TRAIN_27B_MODE = os.environ.get("DD_TRAIN_27B_MODE", "FAST").upper()     # "FAST" or "FULL"
REPRO_RUNS = ["seed0_A_n4000", "seed0_C"]                # archived adapters reproduced by inference in 4b
REPRO_SET = "E1"            # "E1": the 925 essays of Liang et al.'s test set; "all": all 14,272 essays (~20x longer)
REPRO_MAX_LEN, REPRO_BATCH = 512, 16                     # as in a40/train_qwen.py
TRAIN_27B_SET = {"FAST": dict(arms=["A", "C"], seeds=[0], n_per_class=800, max_len=256, score_set="core", per_corpus=None, save_steps=25),
                 "FULL": dict(arms=["A", "C"], seeds=[0], n_per_class=4000, max_len=512, score_set="all", per_corpus=None, save_steps=100)}
DELETE_FP8 = "auto"         # "auto": keep the FP8 files only if the disk has room for both; True / False to force
DEQUANT_DEVICE = "auto"     # "auto": GPU if present, else CPU
MIN_GPU_GIB = 75            # 80 GB cards report ~79 GiB
WORK_27B = pathlib.Path(os.environ.get("DD_27B_WORK", "/content/qwen27b" if os.path.isdir("/content") else "outputs/nb04_27b"))
SAVE_27B_TO_DRIVE = False   # True: results + training checkpoints in Google Drive (survive a runtime reset)
TINY_TEST = os.environ.get("DD_27B_TINY") == "1"         # automated tests only: tiny random model on CPU instead of the 27B
_SIM_OOM = int(os.environ["DD_SIMULATE_OOM_ABOVE"]) if os.environ.get("DD_SIMULATE_OOM_ABOVE") else None   # tests only

FP8_DIR, BF16_DIR, SMALL_DIR = WORK_27B / "fp8", WORK_27B / "bf16_from_fp8", WORK_27B / "hub_small_files"
RES_27B = WORK_27B / "results"
if SAVE_27B_TO_DRIVE and RUN_27B:
    from google.colab import drive
    drive.mount("/content/drive")
    RES_27B = pathlib.Path("/content/drive/MyDrive/detecting-the-detector/nb04_27b")
if TINY_TEST:
    TRAIN_27B_SET = {k: dict(v, n_per_class=16, max_len=32, score_set="core", per_corpus=3, save_steps=25) for k, v in TRAIN_27B_SET.items()}
    REPRO_MAX_LEN = 128
T27 = TRAIN_27B_SET[TRAIN_27B_MODE]
P4_CHECKS = []              # (section, check, PASS / FAIL / SKIPPED)
t_part = time.time()

P4_GO, P4_DELETE_FP8 = False, DELETE_FP8 is True
if not RUN_27B:
    P4_MSG = ("SKIPPED: RUN_27B = False (default). Sections 4a-4c need a GPU with >= 80 GB; see the instructions above. "
              "Only the pre-flight checks of 4.0 run.")
elif TINY_TEST:
    P4_GO, P4_MSG = True, "TINY TEST MODE (automated tests only): a tiny random Qwen3.5 model on the CPU stands in for the 27B. NOT a 27B result."
else:
    ok, gmsg, GPU_INFO = LQ.gpu_check(MIN_GPU_GIB)
    free = LQ.disk_free(WORK_27B)
    if DELETE_FP8 == "auto":
        P4_DELETE_FP8 = free < LQ.disk_needed(FP8_DIR, BF16_DIR, delete_fp8=False)
    need = LQ.disk_needed(FP8_DIR, BF16_DIR, delete_fp8=P4_DELETE_FP8)
    dmsg = (f"disk at {WORK_27B}: {free / 1e9:.0f} GB free, {need / 1e9:.0f} GB needed "
            f"({'FP8 shards deleted after conversion' if P4_DELETE_FP8 else 'FP8 files kept'})")
    if not ok:
        P4_MSG = f"SKIPPED: RUN_27B = True, but {gmsg} Choose Runtime > Change runtime type > RTX PRO 6000 / A100 80 GB / H100."
    elif free < need:
        P4_MSG = f"SKIPPED: RUN_27B = True, but not enough disk: {dmsg}. Set DELETE_FP8 = True or free disk space."
    else:
        P4_GO, P4_MSG = True, f"GO: {gmsg} | {dmsg}"
print(P4_MSG)
if P4_GO:
    WORK_27B.mkdir(parents=True, exist_ok=True); RES_27B.mkdir(parents=True, exist_ok=True)
    print(f"work folder {WORK_27B} | results {RES_27B} | re-training: {'on, ' + TRAIN_27B_MODE if TRAIN_27B else 'off'}")

SKIPPED: RUN_27B = False (default). Sections 4a-4c need a GPU with >= 80 GB; see the instructions above. Only the pre-flight checks of 4.0 run.


### 4.0 Pre-flight checks (no GPU needed; run in every mode)
1. **Conversion formula** on synthetic FP8 numbers with block scales (shapes that are and are not multiples of 128),
   against an independent block-by-block loop and float64 arithmetic (only bf16 rounding allowed); a plausible mistake
   (dividing by the tensor called `weight_scale_inv` instead of multiplying) must give different numbers.
2. **Which tensors are converted**, read from the real repo's index and the 66 file headers (HTTP range requests, a few
   kB each, no weights): every FP8 `*.weight` must have a `*.weight_scale_inv` of the right shape, and the list of
   tensors kept in bf16 is shown.
3. **The real 27B architecture without memory**: the model is built on PyTorch's *meta* device (shapes only), wrapped
   with the LoRA recipe, and compared with the original run: 124,741,120 trainable parameters, only the new head
   `score` missing from the checkpoint, and the same 1,213 tensor names as the archived adapter (from Part 2's header).
4. **The tolerances of 4b can fail**: applied to two *different* archived runs (other seed or other arm), they must
   FAIL for every pair, and each of the four score measures must fail every pair on its own; applied to a run and
   itself they must PASS.

In [17]:
import importlib.util
t_pre = time.time()
# 1. conversion formula on synthetic numbers
try:
    import torch
    g = torch.Generator().manual_seed(0); res = []
    for o, i in [(256, 384), (128, 128), (300, 200), (48, 5120), (5, 7), (130, 257)]:
        w = (torch.randn(o, i, generator=g) * 3).to(torch.float8_e4m3fn)
        s = (torch.rand(-(-o // 128), -(-i // 128), generator=g) * 0.01 + 1e-4).to(torch.bfloat16)
        a = LQ.dequant_fp8_block(w, s); b = LQ.dequant_reference_loop(w, s)
        se = s.float().repeat_interleave(128, 0)[:o].repeat_interleave(128, 1)[:, :i]
        wrong = (w.float() / se).to(torch.bfloat16)        # a plausible mistake: DIVIDING by the 'scale_inv' tensor
        res.append(dict(shape=f"{o} x {i}", multiple_of_128=o % 128 == 0 and i % 128 == 0, dtype=str(a.dtype).replace("torch.", ""),
                        identical_to_loop=bool(torch.equal(a, b)), wrong_variant_detected=bool(not torch.equal(a, wrong)),
                        max_rel_err_vs_float64=float(((a.double() - w.double() * se.double()).abs().max() / (w.double() * se.double()).abs().max()))))
    RES_DQ = pd.DataFrame(res); display(RES_DQ)
    ok = RES_DQ.identical_to_loop.all() and (RES_DQ.dtype == "bfloat16").all() and RES_DQ.wrong_variant_detected.all() \
        and (~RES_DQ.multiple_of_128).any() and (RES_DQ.max_rel_err_vs_float64 <= 2 ** -8).all()
    P4_CHECKS.append(("4.0", "conversion formula = block-by-block reference, bf16 rounding only (6 shapes, 4 not multiples of 128); dividing by the scale is detected", "PASS" if ok else "FAIL"))
except Exception as e:
    print(f"conversion check could not run: {type(e).__name__}: {e}")
    P4_CHECKS.append(("4.0", "conversion formula on synthetic numbers", "FAIL"))

# 2. tensor selection on the real repo (index + headers, no weights)
try:
    from huggingface_hub import snapshot_download
    snapshot_download(LQ.BASE_REPO, revision=LQ.BASE_REVISION, local_dir=str(SMALL_DIR),
                      allow_patterns=["config.json", "model.safetensors.index.json", "tokenizer*", "vocab.json", "merges.txt",
                                      "chat_template.jinja", "generation_config.json", "crc32.txt"])
    WM27, HEADS27 = LQ.shard_headers_from_hub()
    plan_checks, KEPT27, plan_sum = LQ.check_plan(WM27, HEADS27)
    print(f"{LQ.BASE_REPO} @ {LQ.BASE_REVISION[:10]}: {plan_sum['index_entries']:,} index entries in {len(HEADS27)} shards -> "
          f"{plan_sum['dequant']} FP8 tensors converted to bf16, {plan_sum['keep']} kept unchanged, {plan_sum['drop']} scale tensors "
          f"dropped ({plan_sum['non_multiple_of_128']} FP8 tensors with a size that is not a multiple of 128)")
    display(pd.DataFrame(plan_checks, columns=["check", "ok"]))
    print("Tensors kept unchanged (all bf16 in the repo), grouped:")
    with pd.option_context("display.max_rows", 40):
        display(KEPT27)
    ok = all(c for _, c in plan_checks) and plan_sum["dequant"] == 407 and plan_sum["keep"] == 792 and plan_sum["drop"] == 407
    P4_CHECKS.append(("4.0", "real repo: 407 FP8 weights, each with a scale of the right shape; 792 kept; 407 scales dropped", "PASS" if ok else "FAIL"))
except Exception as e:
    print(f"SKIPPED tensor-selection check (needs internet access to huggingface.co): {type(e).__name__}: {str(e)[:150]}")
    P4_CHECKS.append(("4.0", "real repo: tensor selection", "SKIPPED"))

# 3. the real architecture on the meta device
if importlib.util.find_spec("transformers.models.qwen3_5") is None or not (SMALL_DIR / "config.json").exists():
    print("SKIPPED meta-device check: transformers without Qwen3.5 classes, or the config could not be downloaded.")
    P4_CHECKS.append(("4.0", "real 27B architecture on the meta device", "SKIPPED"))
else:
    akeys = HEADERS["seed0_A_n4000"].name.tolist() if HEADERS.get("seed0_A_n4000") is not None else None
    MC = LQ.meta_model_checks(SMALL_DIR, akeys)
    print(f"{MC['model_class']} built on the meta device: {MC['base_params']:,} numbers; with LoRA: {MC['trainable']:,} trainable "
          f"of {MC['total']:,} ({MC['trainable'] / MC['total']:.3%}); model tensors not in the checkpoint: {MC['model_not_in_checkpoint']}; "
          f"checkpoint tensors the model does not use: {len(MC['checkpoint_not_in_model'])} (lm_head + MTP head)")
    P4_CHECKS.append(("4.0", "meta model + LoRA recipe: 124,741,120 trainable parameters (as printed by the original run)",
                      "PASS" if MC["trainable"] == LQ.EXPECTED_TRAINABLE else "FAIL"))
    P4_CHECKS.append(("4.0", "meta model: only 'score.weight' missing from the checkpoint; unused = lm_head + mtp.*",
                      "PASS" if MC["model_not_in_checkpoint"] == ["score.weight"] and all(k == "lm_head.weight" or k.startswith("mtp.")
                                                                                     for k in MC["checkpoint_not_in_model"]) else "FAIL"))
    if akeys is None:
        print("SKIPPED adapter-name comparison: no adapter header from Part 2 (no Hugging Face token).")
        P4_CHECKS.append(("4.0", "meta model: tensor names = archived adapter seed0_A_n4000", "SKIPPED"))
    else:
        print(f"adapter seed0_A_n4000: {len(akeys)} tensor names in the file, {MC['peft_keys']} in the meta model; "
              f"only in file {MC['adapter_keys_only_in_file']}, only in model {MC['adapter_keys_only_in_model']}")
        P4_CHECKS.append(("4.0", "meta model: 1,213 tensor names = archived adapter seed0_A_n4000", "PASS" if MC["adapter_keys_equal"] else "FAIL"))

# 4. the tolerances can fail: every pair of different archived runs, and each run against itself
E1N = 925
arch = {D.run_name(a, s): D.load_preds("Qwen27B", a, s).iloc[:E1N] for a in "ABCD" for s in D.SEEDS["Qwen27B"]}
pairs = [LQ.compare_scores(arch[x], arch[y]) | dict(x=x, y=y) for i, x in enumerate(arch) for y in list(arch)[i + 1:]]
PAIRS = pd.DataFrame(pairs)
selfp = [LQ.compare_scores(arch[x], arch[x])["result"] for x in arch]
closest = PAIRS.sort_values("same_decision_at_0.5").iloc[-1]
print(f"Tolerances {LQ.REPRO_TOL} applied to the {len(PAIRS)} pairs of different archived runs (925 Liang essays): "
      f"{(PAIRS.result == 'FAIL').sum()} FAIL, {(PAIRS.result == 'PASS').sum()} PASS. Closest pair {closest.x} vs {closest.y}: "
      f"Pearson {closest.pearson:.3f}, Spearman {closest.spearman:.3f}, mean |diff| {closest.mean_abs_diff:.3f}, "
      f"same decision {closest['same_decision_at_0.5']:.1%}. Each run against itself: {selfp.count('PASS')} of {len(selfp)} PASS.")
T = LQ.REPRO_TOL
alone = {"Pearson": (PAIRS.pearson >= T["pearson"]).sum(), "Spearman": (PAIRS.spearman >= T["spearman"]).sum(),
         "mean |diff|": (PAIRS.mean_abs_diff <= T["mean_abs_diff"]).sum(), "same decision": (PAIRS["same_decision_at_0.5"] >= T["agree"]).sum()}
print("pairs of different runs that pass ONE measure alone:", alone)
P4_CHECKS.append(("4.0", "tolerances FAIL on all 28 pairs of different archived runs (each of the 4 score measures alone too) and PASS on identical scores",
                  "PASS" if (PAIRS.result == "FAIL").all() and len(PAIRS) == 28 and selfp.count("PASS") == len(selfp)
                  and all(v == 0 for v in alone.values()) else "FAIL"))
TIMES["Part 4.0 pre-flight"] = time.time() - t_pre

,shape,multiple_of_128,dtype,identical_to_loop,wrong_variant_detected,max_rel_err_vs_float64
0,256 x 384,True,bfloat16,True,True,0.002849
1,128 x 128,True,bfloat16,True,True,0.001715
2,300 x 200,False,bfloat16,True,True,0.001865
3,48 x 5120,False,bfloat16,True,True,0.002245
4,5 x 7,False,bfloat16,True,True,0.001210
5,130 x 257,False,bfloat16,True,True,0.002137


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Qwen/Qwen3.8-27B-FP8 @ c3a462cc3d: 1,606 index entries in 66 shards -> 407 FP8 tensors converted to bf16, 792 kept unchanged, 407 scale tensors dropped (0 FP8 tensors with a size that is not a multiple of 128)


,check,ok
0,"every index entry is in a shard header, same s...",True
1,name rule (has *_scale_inv) == dtype rule (sto...,True
2,every FP8 tensor is a 2-D '*.weight' with a '*...,True
3,every *_scale_inv belongs to an FP8 tensor,True
4,scale shapes = ceil(out/128) x ceil(in/128),True
5,kept tensors are not FP8,True


Tensors kept unchanged (all bf16 in the repo), grouped:


,part,tensor,dtype,tensors
0,MTP head (unused),fc.weight,BF16,1
1,MTP head (unused),input_layernorm.weight,BF16,1
2,MTP head (unused),k_norm.weight,BF16,1
3,MTP head (unused),norm.weight,BF16,1
4,MTP head (unused),post_attention_layernorm.weight,BF16,1
5,MTP head (unused),pre_fc_norm_embedding.weight,BF16,1
6,MTP head (unused),pre_fc_norm_hidden.weight,BF16,1
7,MTP head (unused),q_norm.weight,BF16,1
8,language model,A_log,BF16,48
9,language model,conv1d.weight,BF16,48


Qwen3_5ForSequenceClassification built on the meta device: 26,085,340,400 numbers; with LoRA: 124,741,120 trainable of 26,210,081,520 (0.476%); model tensors not in the checkpoint: ['score.weight']; checkpoint tensors the model does not use: 16 (lm_head + MTP head)
adapter seed0_A_n4000: 1213 tensor names in the file, 1213 in the meta model; only in file 0, only in model 0
Tolerances {'pearson': 0.99, 'spearman': 0.98, 'mean_abs_diff': 0.02, 'agree': 0.98, 'toefl_flag_essays': 3} applied to the 28 pairs of different archived runs (925 Liang essays): 28 FAIL, 0 PASS. Closest pair seed1_A_n4000 vs seed1_B_n4000: Pearson 0.935, Spearman 0.976, mean |diff| 0.048, same decision 95.7%. Each run against itself: 8 of 8 PASS.
pairs of different runs that pass ONE measure alone: {'Pearson': np.int64(0), 'Spearman': np.int64(0), 'mean |diff|': np.int64(0), 'same decision': np.int64(0)}


### 4a · Base model: download FP8, convert to bf16
Skipped unless `RUN_27B = True` on a suitable GPU. Already converted shards are skipped, so re-running resumes.

In [18]:
t_4a = time.time(); BASE_READY = False
if not P4_GO:
    print(P4_MSG)
else:
    import torch
    dq_dev = ("cuda" if torch.cuda.is_available() else "cpu") if DEQUANT_DEVICE == "auto" else DEQUANT_DEVICE
    if TINY_TEST:
        remote = WORK_27B / "tiny_remote_fp8"
        if not (remote / "model.safetensors.index.json").exists():
            print("TINY TEST MODE: building a tiny random FP8 checkpoint in the layout of the Hub repo:",
                  LQ.make_tiny_fp8_checkpoint(remote, SMALL_DIR))
        fetch = LQ.folder_fetcher(remote, FP8_DIR)
    else:
        fetch = LQ.hub_fetcher(FP8_DIR)
    # GPU and CPU must give the same bits: check on the first shard that holds FP8 weights (before it may be deleted)
    fetch(["*.json", "*.txt", "*.jinja"])
    wm = json.load(open(FP8_DIR / "model.safetensors.index.json"))["weight_map"]
    first = sorted({f for k, f in wm.items() if k.endswith("_scale_inv")})[0]
    if dq_dev != "cuda":
        print(f"SKIPPED GPU-vs-CPU check: conversion runs on {dq_dev}.")
        P4_CHECKS.append(("4a", "conversion on GPU = on CPU, bit for bit", "SKIPPED"))
    elif (BF16_DIR / first).exists():
        print(f"SKIPPED GPU-vs-CPU check: {first} was converted in an earlier session.")
        P4_CHECKS.append(("4a", "conversion on GPU = on CPU, bit for bit", "SKIPPED"))
    else:
        fetch(sorted(LQ._needed_files(first, wm)))
        a, _ = LQ.dequant_shard(FP8_DIR, first, wm, device="cpu"); b, n = LQ.dequant_shard(FP8_DIR, first, wm, device="cuda")
        same = all(torch.equal(a[k], b[k]) for k in a)
        print(f"{first}: {n} FP8 tensors converted on CPU and on GPU -> {'bit-identical' if same else 'DIFFERENT'}")
        P4_CHECKS.append(("4a", f"conversion on GPU = on CPU, bit for bit ({first})", "PASS" if same else "FAIL")); del a, b
    print(f"converting on {dq_dev}; FP8 shards {'deleted after use' if P4_DELETE_FP8 else 'kept'} ...", flush=True)
    PREP = LQ.prepare_bf16_base(FP8_DIR, BF16_DIR, fetch, delete_fp8=P4_DELETE_FP8, device=dq_dev,
                                log_every=1 if TINY_TEST else 10)
    print(PREP)
    n_plan = sum(not k.endswith("_scale_inv") for k in wm)
    ok = PREP["shards_missing"] == 0 and PREP["total_tensors"] == n_plan and (TINY_TEST or (PREP["fp8_tensors_in_plan"] == 407 and PREP["total_tensors"] == 1199))
    P4_CHECKS.append(("4a", f"bf16 folder complete: {PREP['shards']} shards, {PREP['total_tensors']:,} tensors, {PREP['bf16_bytes'] / 1e9:.1f} GB",
                      "PASS" if ok else "FAIL"))
    BASE_READY = ok
TIMES["Part 4a base model"] = time.time() - t_4a

SKIPPED: RUN_27B = False (default). Sections 4a-4c need a GPU with >= 80 GB; see the instructions above. Only the pre-flight checks of 4.0 run.


### 4b · Reproduction by inference (the strongest evidence)
The bf16 base is loaded on **one** GPU, the archived adapters `seed0_A_n4000` and `seed0_C` are downloaded from Hugging
Face and attached with PEFT (task `SEQ_CLS`; each adapter brings its LoRA matrices and its trained head `score`), and
every tensor of each adapter file is checked to be in the model with exactly the same values. Then the 925 essays of
Liang et al.'s test set (exam E1: 91 TOEFL, 88 Hewlett, 70 college essays, 145 CS224N, 352 GPT-3 essays, plus the
GPT-4-polished TOEFL and simplified Hewlett versions) are scored. They are the first 925 rows of the original scoring
order, so they are scored in the **same batches of 16, with the same 512-token cut and right padding** as in
`a40/train_qwen.py`. Every new score is compared with the archived score of the same essay in `data/preds/qwen/`.

**Tolerances, fixed before any 27B result exists** (we have none; these numbers were chosen from the archived data
only). The new scores cannot be bit-identical: bf16 arithmetic on a different GPU, different kernels and library
versions change the last digits, and a few essays lie close to the 0.5 threshold. They must, however, be far closer
to the archived run than any *other* run is. Over all 28 pairs of different archived runs on these 925 essays, the
closest values are: Pearson 0.950 (arm A vs B, seed 0), Spearman 0.976 and 95.7% identical decisions (arm A vs B,
seed 1), mean |difference| 0.048 (arm A vs B, seed 0); two seeds of the same arm are further apart (arm A: Pearson
0.745, 85.7% identical decisions, 30 TOEFL essays apart). Only about 1% of the archived scores lie within 0.05 of 0.5,
so numerical noise should flip very few decisions. Each of the first four tolerances below is stricter than the
closest pair of different runs, so each one alone already rejects all 28 pairs. PASS needs **all** of:

| measure (new vs archived, same essays) | tolerance |
|---|---|
| Pearson correlation of the scores | >= 0.99 |
| Spearman (rank) correlation | >= 0.98 (half of the scores are below 0.04, where rounding can reorder close values) |
| mean absolute difference | <= 0.02 |
| identical decisions at threshold 0.5 | >= 98% (at most 18 of 925 essays change) |
| TOEFL essays flagged at 0.5 | differ by at most 3 of 91 essays (3.3 points; this one alone does not separate runs: arm C's two seeds differ by 1 essay) |

Section 4.0 showed that these tolerances FAIL for every pair of different archived runs. As a second negative control
the new scores of each adapter are also compared with all other archived runs: only the matching run may PASS.

In [19]:
t_4b = time.time(); REPRO = {}
if not P4_GO:
    print(P4_MSG)
elif not BASE_READY:
    print("SKIPPED: the bf16 base model is not ready (see 4a).")
else:
    import gc, torch
    SUB = D.score_frame()
    if REPRO_SET == "E1":
        assert (SUB.exam.iloc[:925] == "E1").all() and (SUB.exam.iloc[925:] != "E1").all()
        SUB = SUB.iloc[:925]
    ARCH_DIR = pathlib.Path(D.data_dir()) / "preds" / "qwen"
    if TINY_TEST:
        ARCH_DIR = WORK_27B / "tiny_archive"
        if not all((ARCH_DIR / f"{r}.parquet").exists() for r in REPRO_RUNS):
            print("TINY TEST MODE: creating tiny adapters + their 'archived' scores (stand-ins for the Hugging Face adapters)")
            LQ.make_tiny_adapters(BF16_DIR, WORK_27B / "tiny_adapters", ARCH_DIR, REPRO_RUNS, SUB, REPRO_MAX_LEN)
    todo = [r for r in REPRO_RUNS if not (RES_27B / f"repro_{r}_{REPRO_SET}_len{REPRO_MAX_LEN}.parquet").exists()]
    if todo:
        if TINY_TEST:
            ADIRS = {r: WORK_27B / "tiny_adapters" / r for r in REPRO_RUNS}
        elif TOKEN is None:
            ADIRS = None
            print("SKIPPED: no Hugging Face token, so the archived adapters cannot be downloaded (see Part 2).")
        else:
            ADIRS = {r: LQ.download_adapter(r, TOKEN) for r in REPRO_RUNS}
        if ADIRS:
            t = time.time()
            tok27, base27, linfo = LQ.load_base_seqcls(BF16_DIR, device_map={"": 0} if torch.cuda.is_available() else None)
            print(f"base model loaded in {time.time() - t:.0f} s: {type(base27).__name__}, "
                  f"{sum(p.numel() for p in base27.parameters()):,} numbers, dtype {next(base27.parameters()).dtype}, "
                  f"device {next(base27.parameters()).device}, tokenizer padding side '{tok27.padding_side}'"
                  + (f", GPU memory in use {torch.cuda.memory_allocated() / LQ.GiB:.1f} GiB" if torch.cuda.is_available() else ""))
            unexpected_ok = all(k == "lm_head.weight" or k.startswith("mtp.") for k in linfo["unexpected_keys"])
            print(f"loading report: missing {sorted(linfo['missing_keys'])} (new head, expected); unused from the checkpoint: "
                  f"{len(linfo['unexpected_keys'])} tensors (lm_head + MTP head, expected)")
            P4_CHECKS.append(("4b", "base loaded as in train_qwen.py: only 'score.weight' new, only lm_head/mtp unused",
                              "PASS" if sorted(linfo["missing_keys"]) == ["score.weight"] and unexpected_ok else "FAIL"))
            if tok27.padding_side != "right":
                print(f"NOTE: this tokenizer pads on the '{tok27.padding_side}'; our local transformers 5.23 pads Qwen3.5 on the right.")
            pm27 = LQ.attach_adapters(base27, ADIRS)
            for r in todo:
                v = LQ.verify_adapter_loaded(pm27, r, ADIRS[r])
                print(f"{r}: {v['identical']:,} of {v['tensors_in_file']:,} adapter tensors in the model with identical values; "
                      f"missing {v['missing_in_model']}, extra {v['extra_in_model']} -> {v['result']}")
                P4_CHECKS.append(("4b", f"{r}: every adapter tensor loaded with identical values", v["result"]))
                pm27.set_adapter(r); t = time.time()
                sc = LQ.score_texts(pm27, tok27, SUB.text.tolist(), REPRO_MAX_LEN, batch_size=REPRO_BATCH, sort_by_length=False, log_every=20)
                out = SUB[["exam", "corpus", "essay_id"]].copy(); out["score"] = sc
                out["model"] = "Qwen27B REPRODUCED BY INFERENCE" if not TINY_TEST else "TINY TEST MODEL"; out["run"] = r
                out.to_parquet(RES_27B / f"repro_{r}_{REPRO_SET}_len{REPRO_MAX_LEN}.parquet", index=False)
                print(f"{r}: {len(out):,} essays scored in {time.time() - t:.0f} s"
                      + (f", peak GPU memory {torch.cuda.max_memory_allocated() / LQ.GiB:.1f} GiB" if torch.cuda.is_available() else ""), flush=True)
            del pm27, base27; gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    else:
        print("all reproduction scores exist from an earlier session; comparing them again")
    for r in REPRO_RUNS:
        f = RES_27B / f"repro_{r}_{REPRO_SET}_len{REPRO_MAX_LEN}.parquet"
        if f.exists():
            REPRO[r] = pd.read_parquet(f)
    if REPRO:
        archived = {p.stem: pd.read_parquet(p) for p in sorted(ARCH_DIR.glob("seed*.parquet"))}
        rows = []
        for r, new_s in REPRO.items():
            for a, old_s in archived.items():
                rows.append(dict(new=r, archived=a, expected="PASS" if a == r else "FAIL", **LQ.compare_scores(new_s, old_s.loc[old_s.essay_id.isin(new_s.essay_id)])))
        RCMP = pd.DataFrame(rows)
        lab = "TINY TEST MODEL (not the 27B)" if TINY_TEST else "REPRODUCED BY INFERENCE on this GPU vs ARCHIVED (A40 run)"
        print(lab + ": same adapter")
        cols = ["new", "archived", "essays", "pearson", "spearman", "mean_abs_diff", "max_abs_diff", "same_decision_at_0.5",
                "TOEFL_flagged_new_%", "TOEFL_flagged_archived_%", "TOEFL_flag_diff_essays", "result"]
        display(RCMP[RCMP.new == RCMP.archived][cols].round(4))
        print("negative control: the same new scores against the OTHER archived runs (each must FAIL)")
        display(RCMP[RCMP.new != RCMP.archived][["new", "archived", "pearson", "mean_abs_diff", "same_decision_at_0.5", "result"]].round(3))
        for r in REPRO:
            same = RCMP[(RCMP.new == r) & (RCMP.archived == r)]
            P4_CHECKS.append(("4b", f"{r}: new scores vs archived within the tolerances", same.result.iloc[0] if len(same) else "FAIL"))
            others = RCMP[(RCMP.new == r) & (RCMP.archived != r)]
            P4_CHECKS.append(("4b", f"{r}: new scores FAIL against all {len(others)} other archived runs (negative control)",
                              "PASS" if len(others) and (others.result == "FAIL").all() else "FAIL"))
TIMES["Part 4b reproduction by inference"] = time.time() - t_4b

SKIPPED: RUN_27B = False (default). Sections 4a-4c need a GPU with >= 80 GB; see the instructions above. Only the pre-flight checks of 4.0 run.


### 4c · Re-training the 27B (off by default: `TRAIN_27B = False`)
The recipe of `a40/train_qwen.py` (same `ddlib.lora_qwen.train_and_score` function as Part 3): LoRA r 16, alpha 32,
dropout 0.05 on all linear layers, head trained, learning rate 1e-4, 1 epoch, 16 essays per optimizer step, gradient
checkpointing, bf16, seed = run seed. Differences from the original: **one** GPU instead of two (the model is not
split), and the batch per step is picked by a memory probe (4, 2 or 1 essays, with gradient accumulation 4, 8 or 16;
the original used 8 x 2). After the first steps it prints a projected training time.

| mode | runs | training essays | max. tokens | essays scored | estimate (not measured) |
|---|---|---|---|---|---|
| FAST | arms A, C, seed 0 | 1,600 (800 per class, paired design as in Part 3) | 256 | Liang test set + LOCNESS + validation AI essays (1,096) | ~10-30 min per run + loading |
| FULL | arms A, C, seed 0 (edit `TRAIN_27B_SET["FULL"]` for more) | 8,000 (the original training sets) | 512 | all 14,272 | ~1-4 h per run + ~0.5-2 h scoring |

One run at a time; a finished run is skipped when the notebook is run again, and an interrupted run resumes from its
last checkpoint (every 25 / 100 optimizer steps). **A re-trained 27B is a new training run, not a reproduction**:
different hardware and batch layout and, in FAST, less data and shorter texts; compare it with the archived runs
like a third seed (the two archived seeds of arm A differ by 33 points on TOEFL).

In [20]:
t_4c = time.time(); RERUN = {}
if not P4_GO:
    print(P4_MSG)
elif not TRAIN_27B:
    print("SKIPPED: TRAIN_27B = False (default). Set it to True in the Part 4 CONFIG cell to re-train (FAST ~1 h in total, FULL several hours; estimates).")
elif not BASE_READY:
    print("SKIPPED: the bf16 base model is not ready (see 4a).")
else:
    import torch
    RERUN_LABEL = (f"27B RE-RUN, reduced data ({TRAIN_27B_MODE})" if TRAIN_27B_MODE == "FAST" else "27B RE-RUN, original recipe on one GPU (FULL)")
    if TINY_TEST:
        RERUN_LABEL = "TINY TEST MODEL (not the 27B)"
    SCORE_27B = LQ.score_subset(D.score_frame(), T27["score_set"], T27["per_corpus"])
    pool27, man27 = D.load_pool(cleaned=True), D.load_manifests()
    for s in T27["seeds"]:
        XA, yA = D.load_trainset(D.run_name("A", s), pool_cleaned=pool27, manifests=man27)
        XC, yC = D.load_trainset(D.run_name("C", s))
        partC = pd.read_parquet(D.data_dir() / "followup" / "trainsets" / f"seed{s}_C.parquet").part.tolist()
        subs = dict(zip("AC", LQ.paired_subsample_A_C(XA, yA, XC, yC, partC, T27["n_per_class"], seed=s))) if T27["n_per_class"] < 4000 else {}
        for arm in T27["arms"]:
            X, y = subs[arm] if subs else D.load_trainset(D.run_name(arm, s), pool_cleaned=pool27, manifests=man27)
            cfg = dict(model=str(BF16_DIR), base=f"{LQ.BASE_REPO}@{LQ.BASE_REVISION}", arm=arm, seed=s, n_train=len(X),
                       max_len=T27["max_len"], score_set=T27["score_set"], per_corpus=T27["per_corpus"], n_scored=len(SCORE_27B))
            rid = f"Qwen27B_RERUN_{TRAIN_27B_MODE}_{arm}_seed{s}_n{len(X)}_len{T27['max_len']}"
            if LQ.finished(RES_27B, rid, cfg):
                inf = json.load(open(RES_27B / f"{rid}.json"))
                print(f"SKIP {rid}: finished earlier (trained in {inf['train_s']:.0f} s on {inf['gpu'] or inf['device']})")
            else:
                print(f"RUN  {rid}: {len(X):,} training essays ({sum(y):,} AI), max {T27['max_len']} tokens ...", flush=True)
                inf = LQ.train_and_score(X, y, SCORE_27B, cfg, RES_27B, rid, log_every=10,
                                         device_map={"": 0} if torch.cuda.is_available() else None, batch="auto",
                                         save_steps=T27["save_steps"], ckpt_root=RES_27B / "ckpt", adapter_dir=RES_27B / "adapters",
                                         model_label=RERUN_LABEL, simulate_oom_above=_SIM_OOM)
                print(f"     done: load {inf['load_s']:.0f} s, training {inf['train_s']:.0f} s ({inf['optimizer_steps']} steps, batch "
                      f"{inf['per_device_batch']} x {inf['grad_accum']}), scoring {inf['eval_s']:.0f} s, peak GPU memory "
                      f"{inf['peak_gpu_gib']} GiB | loss first {inf['loss_first']:.3f}, last {inf['loss_last']:.3f} | trainable "
                      f"{inf['trainable_params']:,}", flush=True)
            RERUN[(arm, s)] = (pd.read_parquet(RES_27B / f"{rid}.parquet"), inf)
    rows = []
    for (arm, s), (p, inf) in RERUN.items():
        r = M.run_row(p, train_ids); f = M.liang_prf_from_scores(p)
        arc = QR[(QR.arm == arm) & (QR.seed == s)].iloc[0]
        for col, name, k, newv in (("FP_TOEFL", "TOEFL flagged %", 100, r["FP_TOEFL"]), ("FP_Hewlett", "familiar native (Hewlett) %", 100, r["FP_Hewlett"]),
                                   ("FP_nativeOOD_mean", "unfamiliar native %", 100, r["FP_nativeOOD_mean"]), ("f1", "F1 (Liang set)", 1, f["f1"])):
            rows.append({"arm": arm, "seed": s, "measure": name, RERUN_LABEL: newv * k, "ARCHIVED A40 run (same arm, seed)": arc[col] * k})
    if rows:
        RERUN_RES = pd.DataFrame(rows).set_index(["arm", "seed", "measure"])
        print(RERUN_LABEL + " vs ARCHIVED. A new training run, NOT a bit-identical reproduction; compare it like another seed.")
        display(RERUN_RES.round(3))
    bad = [k for k, (p, inf) in RERUN.items() if inf["nan_scores"] or inf["loss_last"] is None or not np.isfinite(inf["loss_last"])]
    P4_CHECKS.append(("4c", f"{len(RERUN)} of {len(T27['seeds']) * len(T27['arms'])} re-training runs finished, every essay scored, no NaN, "
                      f"{LQ.EXPECTED_TRAINABLE:,} trainable" if not TINY_TEST else f"{len(RERUN)} tiny re-training runs finished, no NaN",
                      "PASS" if len(RERUN) == len(T27["seeds"]) * len(T27["arms"]) and not bad
                      and all(len(p) == len(SCORE_27B) for p, _ in RERUN.values())
                      and (TINY_TEST or all(i["trainable_params"] == LQ.EXPECTED_TRAINABLE for _, i in RERUN.values())) else "FAIL"))
TIMES["Part 4c re-training"] = time.time() - t_4c

SKIPPED: RUN_27B = False (default). Sections 4a-4c need a GPU with >= 80 GB; see the instructions above. Only the pre-flight checks of 4.0 run.


In [21]:
P4_SUMMARY = pd.DataFrame(P4_CHECKS, columns=["section", "check", "result"])
with pd.option_context("display.max_rows", 50, "display.max_colwidth", 130):
    display(P4_SUMMARY)
print(f"PART 4 CHECK SUMMARY: {(P4_SUMMARY.result == 'PASS').sum()} PASS, {(P4_SUMMARY.result == 'FAIL').sum()} FAIL, "
      f"{(P4_SUMMARY.result == 'SKIPPED').sum()} SKIPPED | RUN_27B = {RUN_27B}, TRAIN_27B = {TRAIN_27B}"
      + (" | TINY TEST MODE: not a 27B result" if TINY_TEST else "") + f" | {P4_MSG.split(':')[0]}")
for k in [k for k in TIMES if k.startswith("Part 4")]:
    print(f"  {k}: {TIMES[k]:.0f} s")

,section,check,result
0,4.0,"conversion formula = block-by-block reference, bf16 rounding only (6 shapes, 4 not multiples of 128); dividing by the scale is...",PASS
1,4.0,"real repo: 407 FP8 weights, each with a scale of the right shape; 792 kept; 407 scales dropped",PASS
2,4.0,"meta model + LoRA recipe: 124,741,120 trainable parameters (as printed by the original run)",PASS
3,4.0,meta model: only 'score.weight' missing from the checkpoint; unused = lm_head + mtp.*,PASS
4,4.0,"meta model: 1,213 tensor names = archived adapter seed0_A_n4000",PASS
5,4.0,tolerances FAIL on all 28 pairs of different archived runs (each of the 4 score measures alone too) and PASS on identical scores,PASS


PART 4 CHECK SUMMARY: 6 PASS, 0 FAIL, 0 SKIPPED | RUN_27B = False, TRAIN_27B = False | SKIPPED
  Part 4.0 pre-flight: 26 s
  Part 4a base model: 0 s
  Part 4b reproduction by inference: 0 s
  Part 4c re-training: 0 s
